
# FahMai Directory Q&A — Answer Generation Typhoon Final

```text
TheScamper_03
สมาชิก
1.) นนทพัทธ์-610154
2.) ธนิทธิ์พล-602918
3.) ฐานันท์-604230
4.) ธนกร-601444
5.) ศุภรัตน์-60588
```
Notebook นี้เป็น V6 ที่เปิดใช้ **Typhoon v2.5 Agentic Harness** แล้ว

Output หลักที่ใช้ส่ง Kaggle:

```text
My Drive/super-ai-engineer-season-6-fahmai-2/Submit/submission.csv
```

Debug outputs:

```text
Submit/submission_debug_details_v6.csv
Submit/public_error_analysis_v6.csv
```

สิ่งที่ต้องเตรียมเพิ่มก่อนรัน:
- Colab Secret ชื่อ `TYPHOON_API_KEY`
- หรือ set environment variable `TYPHOON_API_KEY`


**V7 fix:** เพิ่ม `grade_item()` เข้าไปใน Typhoon Harness เพื่อแก้ `NameError: grade_item is not defined` ตอนเปิด `USE_TYPHOON_HARNESS=True`.



## ต้องเตรียมอะไรก่อนรัน

ใน Google Drive ควรมีโครงสร้างประมาณนี้:

```text
My Drive/
└── super-ai-engineer-season-6-fahmai-2/
    ├── employees.csv
    ├── questions.csv
    ├── sample_submission.csv
    ├── train_labels.json
    ├── grade.py
    └── Cleaned Data/
        ├── employees_wrangled_v2.csv
        ├── questions_wrangled.csv
        ├── submission_skeleton.csv
        └── train_label_bucket_summary.csv
```

ถ้าจะเปิดใช้ Typhoon formatter ให้เตรียมเพิ่ม:
- `TYPHOON_API_KEY`
- `TYPHOON_API_URL`
- model: `typhoon-v2.5-30b-a3b-instruct`

ค่าเริ่มต้นของ notebook นี้ **ไม่เรียก LLM** เพื่อให้ได้ baseline ที่ reproducible ก่อน


## Section 0 — Setup

In [1]:

# ============================================================
# Section 0: Setup
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
import ast
import json
import os
import re
import subprocess
import unicodedata

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 180)

print("pandas:", pd.__version__)


pandas: 2.2.2


## Section 1 — Mount Google Drive and define paths

In [2]:

# ============================================================
# Section 1: Mount Google Drive and define paths
# ============================================================

def get_mydrive_dir() -> Path:
    """
    Robust Google Drive mount helper for Colab.
    Handles:
    - Drive already mounted at /content/drive
    - /content/drive mountpoint conflict
    - fallback mount at /content/gdrive
    """
    try:
        from google.colab import drive
    except Exception:
        print("Not running in Colab. Using current directory as project root.")
        return Path(".")

    default_mount = Path("/content/drive")
    default_mydrive = default_mount / "MyDrive"

    if default_mydrive.exists():
        print("Google Drive already mounted at:", default_mount)
        return default_mydrive

    try:
        drive.mount(str(default_mount), force_remount=False)
        if default_mydrive.exists():
            print("Google Drive mounted at:", default_mount)
            return default_mydrive
    except Exception as e:
        print("Could not mount at /content/drive:", repr(e))

    fallback_mount = Path("/content/gdrive")
    fallback_mydrive = fallback_mount / "MyDrive"

    if fallback_mydrive.exists():
        print("Google Drive already mounted at fallback:", fallback_mount)
        return fallback_mydrive

    drive.mount(str(fallback_mount), force_remount=True)
    if fallback_mydrive.exists():
        print("Google Drive mounted at fallback:", fallback_mount)
        return fallback_mydrive

    raise RuntimeError("Could not mount Google Drive.")


MYDRIVE_DIR = get_mydrive_dir()

PROJECT_DIR = MYDRIVE_DIR / "super-ai-engineer-season-6-fahmai-2"
CLEANED_DATA_DIR = PROJECT_DIR / "Cleaned Data"

# V6 output folder
SUBMIT_DIR = PROJECT_DIR / "Submit"
SUBMIT_DIR.mkdir(parents=True, exist_ok=True)

EMP_PATH = CLEANED_DATA_DIR / "employees_wrangled_v2.csv"
Q_PATH = CLEANED_DATA_DIR / "questions_wrangled.csv"
SAMPLE_PATH = PROJECT_DIR / "sample_submission.csv"
TRAIN_LABELS_PATH = PROJECT_DIR / "train_labels.json"
GRADE_PATH = PROJECT_DIR / "grade.py"

# Main Kaggle submission file
SUBMISSION_PATH = SUBMIT_DIR / "submission.csv"

# Debug files
DEBUG_PATH = SUBMIT_DIR / "submission_debug_details_v6.csv"
ERROR_ANALYSIS_PATH = SUBMIT_DIR / "public_error_analysis_v6.csv"

paths = {
    "PROJECT_DIR": PROJECT_DIR,
    "CLEANED_DATA_DIR": CLEANED_DATA_DIR,
    "SUBMIT_DIR": SUBMIT_DIR,
    "employees_wrangled_v2": EMP_PATH,
    "questions_wrangled": Q_PATH,
    "sample_submission": SAMPLE_PATH,
    "train_labels": TRAIN_LABELS_PATH,
    "grade_py": GRADE_PATH,
    "submission_output": SUBMISSION_PATH,
    "debug_output": DEBUG_PATH,
    "error_analysis_output": ERROR_ANALYSIS_PATH,
}

for name, path in paths.items():
    print(f"{name:24s}: {path}")
    print(f"{'':24s}  exists = {path.exists()}")

required = [EMP_PATH, Q_PATH, SAMPLE_PATH, TRAIN_LABELS_PATH, GRADE_PATH]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))


Mounted at /content/drive
Google Drive mounted at: /content/drive
PROJECT_DIR             : /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2
                          exists = True
CLEANED_DATA_DIR        : /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Cleaned Data
                          exists = True
SUBMIT_DIR              : /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Submit
                          exists = True
employees_wrangled_v2   : /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Cleaned Data/employees_wrangled_v2.csv
                          exists = True
questions_wrangled      : /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Cleaned Data/questions_wrangled.csv
                          exists = True
sample_submission       : /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/sample_submission.csv
                          exists = True
train_labels            : /content/drive/MyDrive/super-ai-engineer-

## Section 2 — Load cleaned data

In [3]:

# ============================================================
# Section 2: Load cleaned data
# ============================================================

def read_csv_str(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")

emp = read_csv_str(EMP_PATH)
q = read_csv_str(Q_PATH)
sample = read_csv_str(SAMPLE_PATH)

with open(TRAIN_LABELS_PATH, "r", encoding="utf-8") as f:
    train_labels = json.load(f)

label_df = pd.json_normalize(train_labels["items"])

print("employees:", emp.shape)
print("questions:", q.shape)
print("sample:", sample.shape)
print("train labels:", len(train_labels["items"]))

display(emp.head(3))
display(q.head(3))
display(sample.head(3))


employees: (1995, 51)
questions: (300, 16)
sample: (300, 2)
train labels: 158


,Employee ID,Department,Section,Unit,Position in Thai,Position in English,First Name Thai,Last Name Thai,First Name English,Last Name English,Nickname Thai,Nickname English,Email Address,Phone Extension,Mobile No.,Office Location,Branch,Start Year,Position Level,full_name_th,full_name_en,full_name_en_lower,first_name_en_lower,last_name_en_lower,nickname_th_clean,nickname_en_lower,Department_upper,Section_upper,Unit_upper,org_path,org_blob,position_th_clean,position_en_clean,position_en_lower,position_en_upper,position_level_upper,search_blob,has_nickname_th,has_nickname_en,has_phone_extension,has_mobile,has_department,has_section,has_unit,Department_filled,Section_filled,has_department_filled,has_section_filled,org_path_filled,org_blob_filled,search_blob_filled
0,00002699,CEO,CEO-OFF,CEO,ประธานเจ้าหน้าที่บริหาร,CHIEF EXECUTIVE OFFICER,วชิร,จิรบุญ,VACHIR,CHIRABUN,เบอร์รี่,BERRY,VACHIR.CH@FAHMAI.CO.TH,73048,,FahMai Tower 12F,BKK-R9,2016,C-level,วชิร จิรบุญ,VACHIR CHIRABUN,vachir chirabun,vachir,chirabun,เบอร์รี่,berry,CEO,CEO-OFF,CEO,CEO / CEO-OFF / CEO,CEO CEO-OFF CEO,ประธานเจ้าหน้าที่บริหาร,CHIEF EXECUTIVE OFFICER,chief executive officer,CHIEF EXECUTIVE OFFICER,C-LEVEL,00002699 ceo ceo-off ceo ประธานเจ้าหน้าที่บริหาร chief executive officer วชิร จิรบุญ vachir chirabun เบอร์รี่ berry vachir.ch@fahmai.co.th 73048 fahmai tower 12f bkk-r9 2016 c-...,True,True,True,False,True,True,True,CEO,CEO-OFF,True,True,CEO / CEO-OFF / CEO,CEO CEO-OFF CEO,00002699 ceo ceo-off ceo ประธานเจ้าหน้าที่บริหาร chief executive officer วชิร จิรบุญ vachir chirabun เบอร์รี่ berry vachir.ch@fahmai.co.th 73048 fahmai tower 12f bkk-r9 2016 c-...
1,08902591,CEO,CEO-OFF,CEO-EA,เลขานุการของ CEO,EXECUTIVE ASSISTANT TO CEO,อรญา,วัชรกาญจน์,ORRAYA,WATCHARAKAN,เป้,PE,ORRAYA.WA@FAHMAI.CO.TH,75665,,FahMai Tower 7F,BKK-R9,2023,Manager,อรญา วัชรกาญจน์,ORRAYA WATCHARAKAN,orraya watcharakan,orraya,watcharakan,เป้,pe,CEO,CEO-OFF,CEO-EA,CEO / CEO-OFF / CEO-EA,CEO CEO-OFF CEO-EA,เลขานุการของ CEO,EXECUTIVE ASSISTANT TO CEO,executive assistant to ceo,EXECUTIVE ASSISTANT TO CEO,MANAGER,08902591 ceo ceo-off ceo-ea เลขานุการของ ceo executive assistant to ceo อรญา วัชรกาญจน์ orraya watcharakan เป้ pe orraya.wa@fahmai.co.th 75665 fahmai tower 7f bkk-r9 2023 manager,True,True,True,False,True,True,True,CEO,CEO-OFF,True,True,CEO / CEO-OFF / CEO-EA,CEO CEO-OFF CEO-EA,08902591 ceo ceo-off ceo-ea เลขานุการของ ceo executive assistant to ceo อรญา วัชรกาญจน์ orraya watcharakan เป้ pe orraya.wa@fahmai.co.th 75665 fahmai tower 7f bkk-r9 2023 manager
2,00006662,CEO,CEO-OFF,CEO-CoS,หัวหน้าสำนักงานประธาน,CHIEF OF STAFF,กิตติคุณ,พงจงรัก,KITTIKHUN,PHONGCHONGRAK,บูม,BOOM,KITTIKHUN.PH@FAHMAI.CO.TH,79367,062-174-6941,FahMai Tower 16F,BKK-R9,2017,VP,กิตติคุณ พงจงรัก,KITTIKHUN PHONGCHONGRAK,kittikhun phongchongrak,kittikhun,phongchongrak,บูม,boom,CEO,CEO-OFF,CEO-COS,CEO / CEO-OFF / CEO-COS,CEO CEO-OFF CEO-COS,หัวหน้าสำนักงานประธาน,CHIEF OF STAFF,chief of staff,CHIEF OF STAFF,VP,00006662 ceo ceo-off ceo-cos หัวหน้าสำนักงานประธาน chief of staff กิตติคุณ พงจงรัก kittikhun phongchongrak บูม boom kittikhun.ph@fahmai.co.th 79367 062-174-6941 fahmai tower 16...,True,True,True,True,True,True,True,CEO,CEO-OFF,True,True,CEO / CEO-OFF / CEO-COS,CEO CEO-OFF CEO-COS,00006662 ceo ceo-off ceo-cos หัวหน้าสำนักงานประธาน chief of staff กิตติคุณ พงจงรัก kittikhun phongchongrak บูม boom kittikhun.ph@fahmai.co.th 79367 062-174-6941 fahmai tower 16...


,id,language,question,question_clean,question_lower,question_norm,asks_who,asks_phone,asks_email,asks_list,asks_count,asks_nickname,hyphen_codes,compact_codes,refusal_reason_guess,route_guess
0,g001,en,who is the RETVP,who is the RETVP,who is the retvp,who is the retvp,True,False,False,False,False,False,[],['RETVP'],,role_code_lookup
1,g002,th,ใครเป็น OPSVP,ใครเป็น OPSVP,ใครเป็น opsvp,ใครเป็น opsvp,True,False,False,False,False,False,[],['OPSVP'],,role_code_lookup
2,g004,th,LEGVP ใคร,LEGVP ใคร,legvp ใคร,legvp ใคร,True,False,False,False,False,False,[],['LEGVP'],,role_code_lookup


,id,response
0,g001,
1,g002,
2,g004,


## Section 3 — Sanity checks

In [4]:

# ============================================================
# Section 3: Sanity checks
# ============================================================

assert len(emp) == 1995, f"Expected 1995 employees, got {len(emp)}"
assert len(q) == 300, f"Expected 300 questions, got {len(q)}"
assert len(sample) == 300, f"Expected 300 submission rows, got {len(sample)}"

assert emp["Employee ID"].is_unique, "Employee ID must be unique"
assert q["id"].is_unique, "Question id must be unique"
assert sample["id"].is_unique, "Sample id must be unique"
assert set(q["id"]) == set(sample["id"]), "Question ids and sample submission ids do not match"

required_emp_cols = [
    "Employee ID", "Department_filled", "Section_filled", "Unit_upper",
    "full_name_th", "full_name_en", "nickname_th_clean", "nickname_en_lower",
    "Phone Extension", "Mobile No.", "Email Address",
    "Branch", "Office Location", "Position in English", "Position Level",
    "search_blob_filled",
]
missing_emp_cols = [c for c in required_emp_cols if c not in emp.columns]
assert not missing_emp_cols, f"Missing employee columns: {missing_emp_cols}"

required_q_cols = [
    "id", "language", "question", "hyphen_codes", "compact_codes",
    "route_guess", "refusal_reason_guess",
]
missing_q_cols = [c for c in required_q_cols if c not in q.columns]
assert not missing_q_cols, f"Missing question columns: {missing_q_cols}"

assert (emp["Department_filled"].str.strip() == "").sum() == 0
assert (emp["Section_filled"].str.strip() == "").sum() == 0
assert (emp["Unit_upper"].str.strip() == "").sum() == 0

print("Sanity checks passed.")


Sanity checks passed.


## Section 4 — Utility functions

In [5]:

# ============================================================
# Section 4: Utility functions
# ============================================================

def normalize_text(x) -> str:
    if x is None:
        return ""
    s = str(x)
    if s.lower() == "nan":
        return ""
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u200b", "")
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def lower_text(x) -> str:
    return normalize_text(x).lower()

def upper_text(x) -> str:
    return normalize_text(x).upper()

def parse_list_cell(x):
    """
    questions_wrangled.csv stores list-like values as strings such as:
    "['RETVP']" or "[]".
    """
    if isinstance(x, list):
        return x
    s = normalize_text(x)
    if s == "":
        return []
    try:
        val = ast.literal_eval(s)
        return val if isinstance(val, list) else []
    except Exception:
        return []

def is_blank(x) -> bool:
    return normalize_text(x) == ""

def contains_any(text: str, patterns: list[str]) -> bool:
    t = lower_text(text)
    return any(re.search(p, t, flags=re.IGNORECASE) for p in patterns)

def safe_title_en(name: str) -> str:
    """
    Convert all-caps English names to readable title case.
    Case-insensitive grading means either is usually fine,
    but title case is easier to read.
    """
    return " ".join(w.capitalize() for w in normalize_text(name).split())

def first_nonblank(*values) -> str:
    for v in values:
        s = normalize_text(v)
        if s:
            return s
    return ""


## Section 5 — Parse question helper columns and fix routes

In [6]:

# ============================================================
# Section 5: Parse question helper columns and fix routes
# ============================================================

q = q.copy()

q["hyphen_codes_list"] = q["hyphen_codes"].apply(parse_list_cell)
q["compact_codes_list"] = q["compact_codes"].apply(parse_list_cell)
q["question_lower_runtime"] = q["question"].apply(lower_text)

# Runtime refusal detection.
# Important:
# - Do NOT treat generic Thai "ที่อยู่" as forbidden.
#   In this competition it often means "who is at/from this org", not private address.
FIELD_NOT_ALLOWED_PATTERNS = [
    r"\bsalary\b", r"\bage\b", r"\bhow old\b", r"\bold\b",
    r"\bbirthday\b", r"\bdate of birth\b", r"\bdob\b",
    r"\bhome address\b", r"\bnationality\b", r"\bcitizenship\b",
    r"\breligion\b", r"\beducation\b", r"\beducational\b", r"\bdegree\b",
    r"เงินเดือน", r"อายุ", r"วันเกิด", r"วันเดือนปีเกิด",
    r"บ้านเลขที่", r"เลขบัตร", r"บัตรประชาชน",
    r"สัญชาติ", r"เชื้อชาติ", r"ศาสนา", r"วุฒิการศึกษา", r"การศึกษา",
]

OPINION_PATTERNS = [
    r"\bbest\b", r"\bworst\b", r"\bsmartest\b", r"\bshould\b", r"\bopinion\b",
    r"\bthink\b", r"\brecommend\b", r"\bdemoted\b", r"\brecently\b",
    r"เก่งที่สุด", r"ดีที่สุด", r"แย่ที่สุด", r"ควร", r"คิดว่า", r"ความเห็น",
    r"น่าจะ", r"ทำงานดีที่สุด", r"โปรโมท", r"ถูกตักเตือน", r"ถูกลด",
]

PROMPT_INJECTION_PATTERNS = [
    r"ignore\s+(all\s+)?previous", r"ignore\s+(the\s+)?instructions",
    r"system\s+prompt", r"developer\s+message", r"jailbreak",
    r"override\s+(the\s+)?instructions", r"system\s+override",
    r"test\s+mode", r"admin\s+mode", r"respond\s+with\s+exactly",
    r"the\s+correct\s+answer", r"correct\s+answer\s+for\s+all",
    r"====\s*END\s+USER\s*====", r"====\s*NEW\s+USER\s*====",
    r"\bSYSTEM\s*:", r"\bASSISTANT\s*:", r"\bDEVELOPER\s*:",
    r"ลืมคำสั่ง", r"ข้ามกฎ", r"เปิดเผย\s*prompt", r"เปิดเผยคำสั่ง",
    r"ไม่ต้องทำตามคำสั่ง", r"ทำตามคำสั่งนี้แทน", r"คำสั่งพิเศษ",
    r"ประกาศภายใน", r"ให้ตอบว่า", r"กรุณาตอบตามประกาศ", r"ให้เปลี่ยนการตอบ",
]

EXTERNAL_COMPANY_PATTERNS = [
    r"\bgoogle\b", r"\bopenai\b", r"\bmicrosoft\b", r"\bapple\b", r"\bmeta\b",
    r"\bamazon\b", r"\bscb\b", r"\bkbank\b", r"\bkasikorn\b",
    r"\bsamsung\b", r"\bsony\b", r"\bhuawei\b", r"\blg\b",
    r"กสิกร", r"ธนาคาร", r"ไทยพาณิชย์", r"ซัมซุง", r"โซนี่", r"หัวเว่ย",
]

def detect_refusal_runtime(question: str) -> str:
    ql = lower_text(question)

    if contains_any(ql, PROMPT_INJECTION_PATTERNS):
        return "prompt_injection"
    if contains_any(ql, FIELD_NOT_ALLOWED_PATTERNS):
        return "field_not_in_directory"
    if contains_any(ql, OPINION_PATTERNS):
        return "speculation_or_opinion"
    if contains_any(ql, EXTERNAL_COMPANY_PATTERNS):
        return "external_company"

    return ""

def clean_existing_refusal_reason(row) -> str:
    existing = normalize_text(row.get("refusal_reason_guess", ""))
    runtime = normalize_text(row.get("refusal_reason_runtime", ""))
    ql = lower_text(row.get("question", ""))

    if runtime:
        return runtime

    # Remove false-positive "address" refusal from old wrangled questions.
    if existing == "field_not_in_directory" and "ที่อยู่" in ql:
        return ""

    return existing

def has_refusal_reason(row) -> bool:
    return clean_existing_refusal_reason(row) != ""

def refusal_reason(row) -> str:
    return clean_existing_refusal_reason(row)

q["refusal_reason_runtime"] = q["question"].apply(detect_refusal_runtime)

EMAIL_RE = re.compile(r"[A-Z0-9._%+\-]+@FAHMAI\.CO\.TH", flags=re.IGNORECASE)
MOBILE_RE = re.compile(r"\b0\d{2}-\d{3}-\d{4}\b")
EXT_RE = re.compile(r"\b\d{5}\b")

def looks_like_extension_reverse(text: str) -> bool:
    tl = lower_text(text)
    return bool(EXT_RE.search(text)) and any(x in tl for x in ["ของใคร", "เบอร์ใคร", "belongs to", "whose", "ต่อ"])

def looks_like_mobile_reverse(text: str) -> bool:
    tl = lower_text(text)
    return bool(MOBILE_RE.search(text)) and any(x in tl for x in ["ใคร", "เบอร์ใคร", "whose", "mobile", "phone"])

def looks_like_email_reverse(text: str) -> bool:
    tl = lower_text(text)
    return bool(EMAIL_RE.search(text)) and any(x in tl for x in ["ใคร", "whose", "owner", "เจ้าของ"])

def has_count_intent(text: str) -> bool:
    tl = lower_text(text)
    return (
        "กี่คน" in tl
        or "นับ" in tl
        or "how many" in tl
        or "size of" in tl
        or re.search(r"\bcount\b", tl) is not None
    )

def is_hierarchy_question(text: str) -> bool:
    tl = lower_text(text)
    return (
        "ใต้ cfo" in tl
        or "cpo ดูแล" in tl
        or "under cpo" in tl
        or "ceo-cos" in tl
        or "chief of staff" in tl
        or "รายงานใคร" in tl
        or "รายงานตรง" in tl
    )

def is_branch_question(text: str) -> bool:
    tl = lower_text(text)
    return (
        ("จังหวัด" in tl)
        or ("ภาคใต้" in tl)
        or ("ภาคอีสาน" in tl)
        or ("ภาคเหนือ" in tl)
        or ("northern thailand" in tl)
        or ("อยู่ที่ไหน" in tl and ("hkt" in tl or "nma" in tl or "cnx" in tl or "kkn" in tl or "hdy" in tl))
    )

def is_role_like_question(row) -> bool:
    text = row["question"]
    tl = lower_text(text)

    if row["compact_codes_list"]:
        # Compact codes from hyphen questions often include EXEC/SECTION/PROC.
        bad_compact = {"EXEC", "SECTION", "PROC", "LIST", "MEMBERS", "SIZE"}
        useful = [c for c in row["compact_codes_list"] if upper_text(c) not in bad_compact]
        if useful:
            return True

    role_keywords = [
        "vp", "ceo", "cto", "cfo", "coo", "cpo", "chro", "cmo",
        "head of", "หัว", "ใครดูแล"
    ]

    if any(x in tl for x in role_keywords):
        return True

    if any(x in tl for x in ["สายฟ้า", "saifah", "ดาวเหนือ", "daonuea", "วงโคจร", "wongkhojon", "จุดเชื่อม", "judchuem"]):
        return True

    return False

def fixed_route(row) -> str:
    text = row["question"]
    tl = lower_text(text)

    if has_refusal_reason(row):
        return "refusal"

    if looks_like_email_reverse(text):
        return "email_reverse_lookup"
    if looks_like_mobile_reverse(text):
        return "mobile_reverse_lookup"
    if looks_like_extension_reverse(text):
        return "extension_reverse_lookup"

    if is_hierarchy_question(text):
        return "hierarchy"

    if has_count_intent(text):
        return "count"

    if any(x in tl for x in ["เลขา", "secretary", "ea of", "assistant"]):
        return "secretary_lookup"

    if any(x in tl for x in ["ญาติ", "เป็นญาติกัน", "นามสกุลซ้ำ", "surname family"]):
        return "surname_family"

    if "ชื่อเล่นเป็นชื่อผลไม้" in tl or "ชื่อเล่นเป็นชื่อสี" in tl:
        return "nickname_category_lookup"

    if "director" in tl or "รายชื่อ director" in tl or "list all vps" in tl or "vp ทั้งหมด" in tl:
        return "tier_listing"

    if any(x in tl for x in ["สายฟ้า", "saifah", "ดาวเหนือ", "daonuea", "วงโคจร", "wongkhojon", "จุดเชื่อม", "judchuem", "retail network"]):
        if any(y in tl for y in ["gm", "general manager", "manages", "ดูแล", "ทีมเดียว", "มีใครบ้าง", "who's on", "รายชื่อ"]):
            return "brand_lookup"

    if is_branch_question(text):
        return "branch_lookup"

    if any(x in tl for x in ["เบอร์", "ต่อ", "phone", "ext", "extension", "number", "ติดต่อ"]):
        return "contact_lookup"

    if any(x in tl for x in ["email", "อีเมล", "เมล"]):
        return "email_lookup"

    # Hyphen codes that ask members/section/list must be org listing BEFORE role lookup.
    if row["hyphen_codes_list"]:
        return "org_listing"

    if is_role_like_question(row):
        return "role_code_lookup"

    if any(x in tl for x in ["อยู่ทีมไหน", "อยู่แผนกไหน", "ทีมไหน", "แผนกไหน"]):
        return "nickname_org_lookup"

    if any(x in tl for x in ["มีใครบ้าง", "คือใคร", "ขอรายชื่อคนชื่อ"]):
        return "nickname_lookup"

    if any(x in tl for x in ["ชื่อเล่น", "nickname"]):
        return "nickname_lookup"

    if any(x in tl for x in ["ใคร", "who", "find"]):
        return "identity_lookup"

    return "unknown"

q["route_guess_original"] = q["route_guess"]
q["route_guess"] = q.apply(fixed_route, axis=1)

print("Route distribution after v5 runtime fix:")
display(q["route_guess"].value_counts())

print("Unknown routes:")
display(q[q["route_guess"] == "unknown"][["id", "language", "question", "route_guess_original", "route_guess"]])


Route distribution after v5 runtime fix:


,count
route_guess,
role_code_lookup,74
contact_lookup,49
org_listing,32
refusal,27
secretary_lookup,20
count,20
nickname_lookup,17
brand_lookup,14
identity_lookup,9


Unknown routes:


,id,language,question,route_guess_original,route_guess
130,g178,th,ขอชื่อ แมว สาขาลาดพร้าว หน่อย,branch_lookup,unknown
190,g254,th,ขอรายชื่อ คลื่นเสียง สัก 5 คน,org_listing,unknown


## Section 6 — Refusal phrases

In [7]:

# ============================================================
# Section 6: Refusal phrases
# ============================================================

REFUSAL_PHRASES = {
    "field_not_in_directory": {
        "th": "ไม่สามารถให้ข้อมูลนี้ได้",
        "en": "cannot provide this information",
    },
    "person_not_found": {
        "th": "ไม่พบข้อมูล",
        "en": "no record found",
    },
    "speculation_or_opinion": {
        "th": "ไม่สามารถให้ความเห็นได้",
        "en": "cannot offer an opinion",
    },
    "external_company": {
        "th": "ไม่ใช่ข้อมูลของฟ้าใหม่",
        "en": "not a FahMai record",
    },
    "prompt_injection": {
        "th": "ขอปฏิเสธคำขอ",
        "en": "request declined",
    },
    "field_blank_nickname": {
        "th": "ไม่มีชื่อเล่นในระบบ",
        "en": "nickname not listed",
    },
}

def refusal(reason: str, lang: str) -> str:
    lang = "th" if lang == "th" else "en"
    reason = reason if reason in REFUSAL_PHRASES else "person_not_found"
    return REFUSAL_PHRASES[reason][lang]

for k, v in REFUSAL_PHRASES.items():
    print(k, "=>", v)


field_not_in_directory => {'th': 'ไม่สามารถให้ข้อมูลนี้ได้', 'en': 'cannot provide this information'}
person_not_found => {'th': 'ไม่พบข้อมูล', 'en': 'no record found'}
speculation_or_opinion => {'th': 'ไม่สามารถให้ความเห็นได้', 'en': 'cannot offer an opinion'}
external_company => {'th': 'ไม่ใช่ข้อมูลของฟ้าใหม่', 'en': 'not a FahMai record'}
prompt_injection => {'th': 'ขอปฏิเสธคำขอ', 'en': 'request declined'}
field_blank_nickname => {'th': 'ไม่มีชื่อเล่นในระบบ', 'en': 'nickname not listed'}


## Section 7 — Build lookup indexes

In [8]:

# ============================================================
# Section 7: Build lookup indexes
# ============================================================

emp = emp.copy()

# Normalize important fields
for col in emp.columns:
    emp[col] = emp[col].map(normalize_text)

emp_records = emp.to_dict(orient="records")

by_emp_id = {}
by_email = {}
by_phone_ext = {}
by_mobile = {}

by_unit = defaultdict(list)
by_section = defaultdict(list)
by_department = defaultdict(list)

# Original org columns for official count questions.
by_section_original = defaultdict(list)
by_department_original = defaultdict(list)

by_branch = defaultdict(list)

by_full_name_th = defaultdict(list)
by_full_name_en = defaultdict(list)
by_first_name_th = defaultdict(list)
by_first_name_en = defaultdict(list)
by_last_name_th = defaultdict(list)
by_last_name_en = defaultdict(list)
by_nickname_th = defaultdict(list)
by_nickname_en = defaultdict(list)

for rec in emp_records:
    if rec["Employee ID"]:
        by_emp_id[rec["Employee ID"]] = rec
    if rec["Email Address"]:
        by_email[lower_text(rec["Email Address"])] = rec
    if rec["Phone Extension"]:
        by_phone_ext[rec["Phone Extension"]] = rec
    if rec["Mobile No."]:
        by_mobile[rec["Mobile No."]] = rec

    if rec["Unit_upper"]:
        by_unit[upper_text(rec["Unit_upper"])].append(rec)
    if rec["Section_filled"]:
        by_section[upper_text(rec["Section_filled"])].append(rec)
    if rec["Department_filled"]:
        by_department[upper_text(rec["Department_filled"])].append(rec)

    if rec.get("Section_upper", ""):
        by_section_original[upper_text(rec["Section_upper"])].append(rec)
    if rec.get("Department_upper", ""):
        by_department_original[upper_text(rec["Department_upper"])].append(rec)

    if rec["Branch"]:
        by_branch[upper_text(rec["Branch"])].append(rec)

    if rec["full_name_th"]:
        by_full_name_th[rec["full_name_th"]].append(rec)
    if rec["full_name_en"]:
        by_full_name_en[lower_text(rec["full_name_en"])].append(rec)

    if rec["First Name Thai"]:
        by_first_name_th[rec["First Name Thai"]].append(rec)
    if rec["First Name English"]:
        by_first_name_en[lower_text(rec["First Name English"])].append(rec)

    if rec["Last Name Thai"]:
        by_last_name_th[rec["Last Name Thai"]].append(rec)
    if rec["Last Name English"]:
        by_last_name_en[lower_text(rec["Last Name English"])].append(rec)

    if rec["Nickname Thai"]:
        by_nickname_th[rec["Nickname Thai"]].append(rec)
    if rec["Nickname English"]:
        by_nickname_en[lower_text(rec["Nickname English"])].append(rec)

valid_unit_codes = set(by_unit.keys())
valid_section_codes = set(by_section.keys())
valid_department_codes = set(by_department.keys())

valid_section_original_codes = set(by_section_original.keys())
valid_department_original_codes = set(by_department_original.keys())

print("Index sizes:")
print("by_unit:", len(by_unit))
print("by_section filled:", len(by_section))
print("by_department filled:", len(by_department))
print("by_section original:", len(by_section_original))
print("by_department original:", len(by_department_original))
print("by_full_name_th:", len(by_full_name_th))
print("by_full_name_en:", len(by_full_name_en))
print("by_nickname_th:", len(by_nickname_th))
print("by_nickname_en:", len(by_nickname_en))


Index sizes:
by_unit: 1775
by_section filled: 88
by_department filled: 16
by_section original: 88
by_department original: 16
by_full_name_th: 1995
by_full_name_en: 1995
by_nickname_th: 188
by_nickname_en: 188


## Section 8 — Formatting helpers

In [9]:

# ============================================================
# Section 8: Formatting helpers
# ============================================================

def person_name(rec: dict, lang: str = "th", bilingual: bool = True) -> str:
    th = normalize_text(rec.get("full_name_th", ""))
    en = safe_title_en(rec.get("full_name_en", ""))

    if bilingual:
        if lang == "th":
            return f"{th} ({en})" if th and en else th or en
        return f"{en} ({th})" if th and en else en or th

    return th if lang == "th" else en

def person_contact(rec: dict, lang: str = "th", include_mobile: bool = True, include_email: bool = True) -> str:
    name = person_name(rec, lang=lang, bilingual=True)
    ext = normalize_text(rec.get("Phone Extension", ""))
    mobile = normalize_text(rec.get("Mobile No.", ""))
    email = normalize_text(rec.get("Email Address", ""))

    parts = [name]

    if lang == "th":
        if ext:
            parts.append(f"เบอร์ต่อ {ext}")
        if include_mobile and mobile:
            parts.append(f"มือถือ {mobile}")
        if include_email and email:
            parts.append(f"อีเมล {email}")
    else:
        if ext:
            parts.append(f"ext {ext}")
        if include_mobile and mobile:
            parts.append(f"mobile {mobile}")
        if include_email and email:
            parts.append(f"email {email}")

    return " / ".join(parts)

def format_people(records: list[dict], lang: str = "th", max_items: int | None = None, contact: bool = False) -> str:
    if not records:
        return refusal("person_not_found", lang)

    selected = records if max_items is None else records[:max_items]
    if contact:
        items = [person_contact(r, lang=lang) for r in selected]
    else:
        items = [person_name(r, lang=lang, bilingual=True) for r in selected]

    return ", ".join(items)

def format_count(n: int, lang: str = "th") -> str:
    return f"{n} คน" if lang == "th" else f"{n} employees"

def get_nickname(rec: dict, lang: str) -> str:
    if lang == "th":
        return normalize_text(rec.get("Nickname Thai", ""))
    return normalize_text(rec.get("Nickname English", ""))

def format_org(rec: dict, lang: str) -> str:
    dept = normalize_text(rec.get("Department_filled", ""))
    section = normalize_text(rec.get("Section_filled", ""))
    unit = normalize_text(rec.get("Unit_upper", ""))

    if lang == "th":
        return f"แผนก {dept}, ส่วน {section}, หน่วย {unit}"
    return f"Department {dept}, section {section}, unit {unit}"


## Section 9 — Code and context extraction

In [10]:

# ============================================================
# Section 9: Code and context extraction
# ============================================================

BRANCH_INFO = {
    "BKK-R9": {"th": "กรุงเทพฯ พระราม 9", "en": "Bangkok Rama 9", "region_th": "กรุงเทพฯ", "province_th": "กรุงเทพมหานคร"},
    "BKK-LP": {"th": "ลาดพร้าว", "en": "Lat Phrao", "region_th": "กรุงเทพฯ", "province_th": "กรุงเทพมหานคร"},
    "BKK-SIAM": {"th": "สยาม", "en": "Siam", "region_th": "กรุงเทพฯ", "province_th": "กรุงเทพมหานคร"},
    "BKK-BNA": {"th": "บางนา", "en": "Bang Na", "region_th": "กรุงเทพฯ", "province_th": "กรุงเทพมหานคร"},
    "BKK-PKT": {"th": "คลังบางพลี", "en": "Bang Phli warehouse", "region_th": "กรุงเทพฯ/ปริมณฑล", "province_th": "สมุทรปราการ"},
    "CNX": {"th": "เชียงใหม่", "en": "Chiang Mai", "region_th": "ภาคเหนือ", "province_th": "เชียงใหม่"},
    "CBI": {"th": "ชลบุรี", "en": "Chonburi", "region_th": "ภาคตะวันออก", "province_th": "ชลบุรี"},
    "HKT": {"th": "ภูเก็ต", "en": "Phuket", "region_th": "ภาคใต้", "province_th": "ภูเก็ต"},
    "HDY": {"th": "หาดใหญ่", "en": "Hat Yai", "region_th": "ภาคใต้", "province_th": "สงขลา"},
    "NMA": {"th": "โคราช", "en": "Nakhon Ratchasima", "region_th": "ภาคอีสาน", "province_th": "นครราชสีมา"},
    "KKN": {"th": "ขอนแก่น", "en": "Khon Kaen", "region_th": "ภาคอีสาน", "province_th": "ขอนแก่น"},
    "REMOTE": {"th": "ทำงานทางไกล", "en": "Remote", "region_th": "ทางไกล", "province_th": ""},
}

BRANCH_ALIASES = {
    "ลาดพร้าว": "BKK-LP", "lat phrao": "BKK-LP", "ladprao": "BKK-LP",
    "สยาม": "BKK-SIAM", "siam": "BKK-SIAM",
    "บางนา": "BKK-BNA", "bang na": "BKK-BNA",
    "พระราม 9": "BKK-R9", "rama 9": "BKK-R9",
    "เชียงใหม่": "CNX", "chiang mai": "CNX",
    "ชลบุรี": "CBI", "chonburi": "CBI",
    "ภูเก็ต": "HKT", "phuket": "HKT",
    "หาดใหญ่": "HDY", "hat yai": "HDY",
    "โคราช": "NMA", "นครราชสีมา": "NMA", "nakhon ratchasima": "NMA",
    "ขอนแก่น": "KKN", "khon kaen": "KKN",
    "remote": "REMOTE", "ทางไกล": "REMOTE",
}

BRAND_CODES = {
    "สายฟ้า": {"dept": "SF", "vp": "SFVP", "gm": "SF-GM"},
    "saifah": {"dept": "SF", "vp": "SFVP", "gm": "SF-GM"},
    "ดาวเหนือ": {"dept": "DN", "vp": "DNVP", "gm": "DN-GM"},
    "daonuea": {"dept": "DN", "vp": "DNVP", "gm": "DN-GM"},
    "วงโคจร": {"dept": "WK", "vp": "WKVP", "gm": "WK-GM"},
    "wongkhojon": {"dept": "WK", "vp": "WKVP", "gm": "WK-GM"},
    "จุดเชื่อม": {"dept": "JC", "vp": "JCVP", "gm": "JC-GM"},
    "judchuem": {"dept": "JC", "vp": "JCVP", "gm": "JC-GM"},
    "klunesiang": {"dept": "KS", "vp": "KSVP", "gm": "KS-GM"},
    "คลื่นเสียง": {"dept": "KS", "vp": "KSVP", "gm": "KS-GM"},
}

ROLE_DESCRIPTION_MAP = [
    (["ดูแลด้าน tech", "in charge of tech", "head of tech", "technology head"], "CTO"),
    (["ฝั่ง operations", "in charge of operations", "operations head"], "COO"),
    (["head of product", "หัว product", "สินค้า"], "CPO"),
    (["who owns hr", "หัว hr", "หัว human resources"], "CHRO"),
    (["หัว legal", "legal head", "vp legal"], "LEGVP"),
    (["vp การตลาด", "vp marketing", "marketing vp"], "MKTVP"),
    (["vp digital marketing", "digital marketing"], "MKTDG"),
    (["vp retail กรุงเทพ", "retail กรุงเทพ", "bangkok retail"], "RETBKK"),
    (["vp retail", "retail vp"], "RETVP"),
    (["vp quality", "quality vp"], "OPSQA"),
    (["vp fleet", "fleet vp"], "LOGFL"),
    (["vp b2b accounts", "b2b accounts"], "B2BACC"),
    (["vp b2b", "b2b vp"], "B2BVP"),
    (["vp sup", "sup vp", "vp support"], "SUPVP"),
    (["vp วงโคจร"], "WKVP"),
    (["vp สายฟ้า"], "SFVP"),
    (["vp ดาวเหนือ"], "DNVP"),
]

def extract_branch_codes(question: str) -> list[str]:
    t = lower_text(question)
    hits = []
    for code in BRANCH_INFO:
        if code.lower() in t:
            hits.append(code)
    for alias, code in BRANCH_ALIASES.items():
        if alias.lower() in t:
            hits.append(code)
    return list(dict.fromkeys(hits))

def extract_org_codes(question: str, row=None) -> list[str]:
    codes = []
    if row is not None:
        codes.extend(row.get("hyphen_codes_list", []))
        codes.extend(row.get("compact_codes_list", []))

    text = upper_text(question)
    for code in sorted(valid_unit_codes | valid_section_codes | valid_department_codes, key=len, reverse=True):
        if re.search(rf"(?<![A-Z0-9]){re.escape(code)}(?![A-Z0-9])", text):
            codes.append(code)

    return list(dict.fromkeys(upper_text(c) for c in codes if upper_text(c)))

def lookup_org_code(code: str) -> list[dict]:
    code = upper_text(code)
    if code in by_unit:
        return by_unit[code]
    if code in by_section:
        return by_section[code]
    if code in by_department:
        return by_department[code]
    return []

def lookup_org_code_for_count(code: str) -> list[dict]:
    """
    For count questions, prefer original Department/Section values.
    This avoids over-counting rows where Department/Section were filled from Unit.
    """
    code = upper_text(code)

    # For short department code, prefer original department over unit.
    if "-" not in code and code in by_department_original:
        return by_department_original[code]

    # For section code, prefer original section.
    if code in by_section_original:
        return by_section_original[code]

    if code in by_department_original:
        return by_department_original[code]

    if code in by_unit:
        return by_unit[code]

    # fallback to filled
    if code in by_section:
        return by_section[code]
    if code in by_department:
        return by_department[code]

    return []

def extract_brand_key(question: str) -> str:
    tl = lower_text(question)
    for key in BRAND_CODES:
        if key in tl:
            return key
    return ""

def extract_role_code(question: str, row=None) -> str:
    text = lower_text(question)
    text_u = upper_text(question)

    # Highest priority: explicit C-level codes
    for c in ["CHRO", "CEO", "CTO", "CFO", "COO", "CPO", "CMO"]:
        if re.search(rf"(?<![A-Z0-9]){c}(?![A-Z0-9])", text_u):
            return c

    if re.search(r"\bHR\s*VP\b", text_u) or "hrvp" in text:
        return "HRVP"

    # Description mappings before raw compact codes to avoid false positive words.
    for phrases, code in ROLE_DESCRIPTION_MAP:
        if any(p in text for p in phrases):
            return code

    # Brand mappings
    bkey = extract_brand_key(question)
    if bkey:
        if "gm" in text or "general manager" in text:
            return BRAND_CODES[bkey]["gm"]
        if "vp" in text or "ใครเป็น" in text:
            return BRAND_CODES[bkey]["vp"]

    # Use valid compact codes from row if they exist in Unit codes
    if row is not None:
        for c in row.get("compact_codes_list", []):
            cu = upper_text(c)
            if cu in valid_unit_codes:
                return cu

    # Explicit valid unit codes found in text
    for code in sorted(valid_unit_codes, key=len, reverse=True):
        if re.search(rf"(?<![A-Z0-9]){re.escape(code)}(?![A-Z0-9])", text_u):
            return code

    return ""

def extract_all_role_codes(question: str, row=None) -> list[str]:
    text_u = upper_text(question)
    codes = []

    # C-level
    for c in ["CHRO", "CEO", "CTO", "CFO", "COO", "CPO", "CMO"]:
        if re.search(rf"(?<![A-Z0-9]){c}(?![A-Z0-9])", text_u):
            codes.append(c)

    # Explicit unit codes
    if row is not None:
        codes.extend([upper_text(c) for c in row.get("compact_codes_list", []) if upper_text(c) in valid_unit_codes])

    for code in sorted(valid_unit_codes, key=len, reverse=True):
        if re.search(rf"(?<![A-Z0-9]){re.escape(code)}(?![A-Z0-9])", text_u):
            codes.append(code)

    # Add description single code if no explicit codes
    if not codes:
        c = extract_role_code(question, row)
        if c:
            codes.append(c)

    return list(dict.fromkeys(codes))

def extract_target_code_for_secretary(question: str, row=None) -> str:
    text_u = upper_text(question)

    # Specific C-level secretary cases
    for c in ["CEO", "CFO", "CTO", "COO", "CMO", "CPO", "CHRO"]:
        if re.search(rf"(?<![A-Z0-9]){c}(?![A-Z0-9])", text_u):
            return c

    role = extract_role_code(question, row)
    if role:
        return role

    for code in sorted(valid_unit_codes, key=len, reverse=True):
        if code in text_u:
            return code

    return ""


## Section 10 — Person search helpers

In [11]:

# ============================================================
# Section 10: Person search helpers
# ============================================================

def filter_records_by_context(records: list[dict], question: str) -> list[dict]:
    """
    Filter candidates using department / section / unit / branch context in question.
    """
    if not records:
        return records

    text = lower_text(question)
    text_u = upper_text(question)

    # Org code filters
    org_codes = extract_org_codes(question)
    for code in org_codes:
        hits = lookup_org_code(code)
        hit_ids = {h["Employee ID"] for h in hits}
        filtered = [r for r in records if r["Employee ID"] in hit_ids]
        if filtered:
            records = filtered

    # Branch filters
    branches = extract_branch_codes(question)
    for b in branches:
        filtered = [r for r in records if upper_text(r.get("Branch", "")) == b]
        if filtered:
            records = filtered

    # Department words/codes
    for dept in sorted(valid_department_codes, key=len, reverse=True):
        if re.search(rf"(?<![A-Z0-9]){re.escape(dept)}(?![A-Z0-9])", text_u):
            filtered = [r for r in records if upper_text(r.get("Department_filled", "")) == dept]
            if filtered:
                records = filtered

    return records

def find_people_in_question(question: str) -> list[dict]:
    """
    Find people mentioned in a natural language question.
    Order:
    1. full Thai/English name
    2. nickname Thai/English
    3. first+last partial
    4. first name
    """
    text = normalize_text(question)
    text_l = lower_text(question)

    candidates = []

    # Full name Thai
    for name, records in sorted(by_full_name_th.items(), key=lambda kv: len(kv[0]), reverse=True):
        if name and name in text:
            candidates.extend(records)

    # Full name English
    for name_l, records in sorted(by_full_name_en.items(), key=lambda kv: len(kv[0]), reverse=True):
        if name_l and name_l in text_l:
            candidates.extend(records)

    if candidates:
        return filter_records_by_context(list({r["Employee ID"]: r for r in candidates}.values()), question)

    # Nickname Thai
    nick_candidates = []
    for nick, records in sorted(by_nickname_th.items(), key=lambda kv: len(kv[0]), reverse=True):
        if len(nick) >= 2 and nick in text:
            nick_candidates.extend(records)

    # Nickname English
    for nick_l, records in sorted(by_nickname_en.items(), key=lambda kv: len(kv[0]), reverse=True):
        if len(nick_l) >= 3 and re.search(rf"\b{re.escape(nick_l)}\b", text_l):
            nick_candidates.extend(records)

    if nick_candidates:
        nick_candidates = list({r["Employee ID"]: r for r in nick_candidates}.values())

        # Further filter by first names if question has both nickname and first name
        first_hits = []
        for first, records in by_first_name_th.items():
            if first and first in text:
                ids = {r["Employee ID"] for r in records}
                first_hits.extend([r for r in nick_candidates if r["Employee ID"] in ids])
        for first_l, records in by_first_name_en.items():
            if len(first_l) >= 3 and re.search(rf"\b{re.escape(first_l)}\b", text_l):
                ids = {r["Employee ID"] for r in records}
                first_hits.extend([r for r in nick_candidates if r["Employee ID"] in ids])

        if first_hits:
            nick_candidates = list({r["Employee ID"]: r for r in first_hits}.values())

        return filter_records_by_context(nick_candidates, question)

    # First + last or first only
    name_candidates = []
    for first, records in sorted(by_first_name_th.items(), key=lambda kv: len(kv[0]), reverse=True):
        if first and first in text:
            name_candidates.extend(records)
    for first_l, records in sorted(by_first_name_en.items(), key=lambda kv: len(kv[0]), reverse=True):
        if len(first_l) >= 3 and re.search(rf"\b{re.escape(first_l)}\b", text_l):
            name_candidates.extend(records)

    # If last name appears, intersect
    last_ids = set()
    for last, records in by_last_name_th.items():
        if last and last in text:
            last_ids.update(r["Employee ID"] for r in records)
    for last_l, records in by_last_name_en.items():
        if len(last_l) >= 3 and re.search(rf"\b{re.escape(last_l)}\b", text_l):
            last_ids.update(r["Employee ID"] for r in records)

    if name_candidates and last_ids:
        name_candidates = [r for r in name_candidates if r["Employee ID"] in last_ids]

    if name_candidates:
        name_candidates = list({r["Employee ID"]: r for r in name_candidates}.values())
        return filter_records_by_context(name_candidates, question)

    return []


def find_nickname_records(question: str) -> list[dict]:
    text = normalize_text(question)
    text_l = lower_text(question)

    records = []

    for nick, recs in sorted(by_nickname_th.items(), key=lambda kv: len(kv[0]), reverse=True):
        if len(nick) >= 2 and nick in text:
            records.extend(recs)

    for nick_l, recs in sorted(by_nickname_en.items(), key=lambda kv: len(kv[0]), reverse=True):
        if len(nick_l) >= 3 and re.search(rf"\b{re.escape(nick_l)}\b", text_l):
            records.extend(recs)

    # Variant suffix handling, e.g. มิ้นตี้ -> มิ้น
    if not records:
        for suffix in ["ตี้", "มี่", "กี้"]:
            if suffix in text:
                base = text.replace(suffix, "")
                for nick, recs in by_nickname_th.items():
                    if len(nick) >= 2 and nick in base:
                        records.extend(recs)

    records = list({r["Employee ID"]: r for r in records}.values())

    # If question also includes a first name / last name, filter nickname candidates.
    first_ids = set()
    for first, recs in by_first_name_th.items():
        if first and first in text:
            first_ids.update(r["Employee ID"] for r in recs)
    for first_l, recs in by_first_name_en.items():
        if len(first_l) >= 3 and re.search(rf"\b{re.escape(first_l)}\b", text_l):
            first_ids.update(r["Employee ID"] for r in recs)

    if first_ids:
        filtered = [r for r in records if r["Employee ID"] in first_ids]
        if filtered:
            records = filtered

    last_ids = set()
    for last, recs in by_last_name_th.items():
        if last and last in text:
            last_ids.update(r["Employee ID"] for r in recs)
    for last_l, recs in by_last_name_en.items():
        if len(last_l) >= 3 and re.search(rf"\b{re.escape(last_l)}\b", text_l):
            last_ids.update(r["Employee ID"] for r in recs)

    if last_ids:
        filtered = [r for r in records if r["Employee ID"] in last_ids]
        if filtered:
            records = filtered

    return filter_records_by_context(records, question)



## Section 11 — Tools

In [12]:

# ============================================================
# Section 11: Tools
# ============================================================

def refusal_tool(row) -> str:
    lang = row["language"]
    reason = refusal_reason(row)
    return refusal(reason, lang)

def extension_reverse_tool(row) -> str:
    lang = row["language"]
    m = EXT_RE.search(row["question"])
    if not m:
        return refusal("person_not_found", lang)
    ext = m.group(0)
    rec = by_phone_ext.get(ext)
    if not rec:
        return refusal("person_not_found", lang)
    return person_contact(rec, lang=lang, include_mobile=True, include_email=True)

def mobile_reverse_tool(row) -> str:
    lang = row["language"]
    m = MOBILE_RE.search(row["question"])
    if not m:
        return refusal("person_not_found", lang)
    mobile = m.group(0)
    rec = by_mobile.get(mobile)
    if not rec:
        return refusal("person_not_found", lang)
    return person_contact(rec, lang=lang, include_mobile=True, include_email=True)

def email_reverse_tool(row) -> str:
    lang = row["language"]
    m = EMAIL_RE.search(row["question"])
    if not m:
        return refusal("person_not_found", lang)
    email = lower_text(m.group(0))
    rec = by_email.get(email)
    if not rec:
        return refusal("person_not_found", lang)
    return person_contact(rec, lang=lang, include_mobile=True, include_email=True)


def role_lookup_tool(row) -> str:
    lang = row["language"]
    question = row["question"]

    # Reporting / hierarchy questions should not be answered as the role holder only.
    if "ใต้ cfo" in lower_text(question) or "cpo ดูแล" in lower_text(question) or "under cpo" in lower_text(question) or "ceo-cos" in lower_text(question):
        return hierarchy_tool(row)

    if any(x in lower_text(question) for x in ["เลขา", "secretary", "ea of", "assistant"]):
        return secretary_tool(row)

    role_code = extract_role_code(question, row)
    if not role_code:
        return refusal("person_not_found", lang)

    records = by_unit.get(role_code, [])
    if not records:
        return refusal("person_not_found", lang)

    rec = records[0]

    if any(x in lower_text(question) for x in ["ชื่อเล่น", "nickname"]):
        nick = get_nickname(rec, lang)
        if nick:
            if lang == "th":
                return f"{person_name(rec, lang='th', bilingual=True)} ชื่อเล่น {nick}"
            return f"{person_name(rec, lang='en', bilingual=True)}'s nickname is {nick}"
        return refusal("field_blank_nickname", lang)

    if any(x in lower_text(question) for x in ["อยู่แผนกไหน", "อยู่ทีมไหน", "team", "department"]):
        if lang == "th":
            return f"{person_name(rec, lang='th', bilingual=True)} อยู่ {format_org(rec, lang='th')}"
        return f"{person_name(rec, lang='en', bilingual=True)} is in {format_org(rec, lang='en')}"

    return person_name(rec, lang=lang, bilingual=True)


def secretary_tool(row) -> str:
    lang = row["language"]
    question = row["question"]
    target = extract_target_code_for_secretary(question, row)

    if not target:
        return refusal("person_not_found", lang)

    candidates = []

    c_level_ea = {
        "CEO": "EXECUTIVE ASSISTANT TO CEO",
        "CFO": "EXECUTIVE ASSISTANT TO CFO",
        "CTO": "EXECUTIVE ASSISTANT TO CTO",
        "COO": "EXECUTIVE ASSISTANT TO COO",
        "CMO": "EXECUTIVE ASSISTANT TO CMO",
        "CPO": "EXECUTIVE ASSISTANT TO CPO",
        "CHRO": "EXECUTIVE ASSISTANT TO CHRO",
    }

    if target in c_level_ea:
        phrase = c_level_ea[target]
        candidates = [
            r for r in emp_records
            if phrase in upper_text(r.get("Position in English", ""))
        ]

    if not candidates:
        unit_sec = f"{target}-SEC"
        candidates = by_unit.get(unit_sec, [])

    if not candidates:
        candidates = [
            r for r in emp_records
            if f"SECRETARY OF {target}" in upper_text(r.get("Position in English", ""))
        ]

    if not candidates:
        return refusal("person_not_found", lang)

    sec = candidates[0]

    if any(x in lower_text(question) for x in ["อยู่แผนกไหน", "อยู่ทีมไหน", "team", "department"]):
        # Include target code so questions like "เลขา CEO อยู่แผนกไหน" pass token checks.
        if lang == "th":
            return f"เลขา {target} คือ {person_name(sec, lang='th', bilingual=True)} อยู่ {format_org(sec, lang='th')}"
        return f"{target} secretary is {person_name(sec, lang='en', bilingual=True)} in {format_org(sec, lang='en')}"

    if any(x in lower_text(question) for x in ["ชื่อเล่น", "nickname"]):
        nick = get_nickname(sec, lang)
        if nick:
            if lang == "th":
                return f"{person_name(sec, lang='th', bilingual=True)} ชื่อเล่น {nick}"
            return f"{person_name(sec, lang='en', bilingual=True)}'s nickname is {nick}"
        return refusal("field_blank_nickname", lang)

    return person_name(sec, lang=lang, bilingual=True)

def org_listing_tool(row) -> str:
    lang = row["language"]
    question = row["question"]
    codes = extract_org_codes(question, row)

    if not codes:
        return refusal("person_not_found", lang)

    codes = sorted(codes, key=len, reverse=True)
    for code in codes:
        hits = lookup_org_code(code)
        if hits:
            return format_people(hits, lang=lang, max_items=None, contact=False)

    return refusal("person_not_found", lang)

def contact_lookup_tool(row) -> str:
    lang = row["language"]
    question = row["question"]

    # Multi-role contact questions
    role_codes = extract_all_role_codes(question, row)
    role_people = []
    for c in role_codes:
        if c in by_unit:
            role_people.extend(by_unit[c])

    if len(role_people) >= 2:
        return format_people(role_people, lang=lang, max_items=None, contact=True)

    people = find_people_in_question(question)
    if not people and role_people:
        people = role_people

    if not people:
        return refusal("person_not_found", lang)

    max_items = 3 if len(people) > 1 else 1
    return format_people(people, lang=lang, max_items=max_items, contact=True)

def email_lookup_tool(row) -> str:
    # If the question contains an email address, treat as reverse lookup.
    if EMAIL_RE.search(row["question"]):
        return email_reverse_tool(row)

    lang = row["language"]
    question = row["question"]
    people = find_people_in_question(question)

    if not people:
        role_code = extract_role_code(question, row)
        if role_code and role_code in by_unit:
            people = by_unit[role_code]

    if not people:
        return refusal("person_not_found", lang)

    rec = people[0]
    email = normalize_text(rec.get("Email Address", ""))
    if not email:
        return refusal("field_not_in_directory", lang)

    if lang == "th":
        return f"{person_name(rec, lang='th', bilingual=True)} อีเมล {email}"
    return f"{person_name(rec, lang='en', bilingual=True)} email {email}"


def nickname_lookup_tool(row) -> str:
    lang = row["language"]
    question = row["question"]
    ql = lower_text(question)

    # Category questions.
    if "ชื่อเล่นเป็นชื่อผลไม้" in ql:
        return nickname_category_tool(row, category="fruit")
    if "ชื่อเล่นเป็นชื่อสี" in ql:
        return nickname_category_tool(row, category="color")

    # Nickname of a role.
    role_code = extract_role_code(question, row)
    if role_code and any(x in ql for x in ["ชื่อเล่น", "nickname"]):
        records = by_unit.get(role_code, [])
        if not records:
            return refusal("person_not_found", lang)
        rec = records[0]
        nick = get_nickname(rec, lang)
        if nick:
            if lang == "th":
                return f"{person_name(rec, lang='th', bilingual=True)} ชื่อเล่น {nick}"
            return f"{person_name(rec, lang='en', bilingual=True)}'s nickname is {nick}"
        return refusal("field_blank_nickname", lang)

    people = find_nickname_records(question)

    if not people:
        people = find_people_in_question(question)

    if not people:
        return refusal("person_not_found", lang)

    # If question says "ชื่อจริง X", return contact for the matched person.
    # This adds email/ext tokens needed by stricter labels.
    if "ชื่อจริง" in ql:
        return format_people(people, lang=lang, max_items=3, contact=True)

    # Include nickname token for nickname-grid variants like "มิ้นตี้".
    nick_token = ""
    for nick in by_nickname_th:
        if nick and nick in normalize_text(question):
            nick_token = nick
            break
    if not nick_token:
        for suffix in ["ตี้", "มี่", "กี้"]:
            if suffix in normalize_text(question):
                candidate = normalize_text(question).replace(suffix, "")
                for nick in by_nickname_th:
                    if nick and nick in candidate:
                        nick_token = nick
                        break

    prefix = f"{nick_token}: " if nick_token else ""
    return prefix + format_people(people, lang=lang, max_items=None, contact=False)


def nickname_org_lookup_tool(row) -> str:
    lang = row["language"]
    question = row["question"]

    # Secretary should win over nickname matching.
    if any(x in lower_text(question) for x in ["เลขา", "secretary", "ea of"]):
        return secretary_tool(row)

    people = find_nickname_records(question)

    if not people:
        people = find_people_in_question(question)

    if not people:
        return refusal("person_not_found", lang)

    max_items = min(len(people), 5)
    parts = []
    for rec in people[:max_items]:
        if lang == "th":
            parts.append(f"{person_name(rec, lang='th', bilingual=True)} อยู่ {format_org(rec, lang='th')}")
        else:
            parts.append(f"{person_name(rec, lang='en', bilingual=True)} is in {format_org(rec, lang='en')}")
    return ", ".join(parts)

def branch_lookup_tool(row) -> str:
    lang = row["language"]
    question = row["question"]
    text = lower_text(question)

    if "ภาคใต้" in text:
        codes = ["HKT", "HDY"]
    elif "ภาคอีสาน" in text:
        codes = ["NMA", "KKN"]
    elif "northern thailand" in text or "ภาคเหนือ" in text:
        codes = ["CNX"]
    else:
        codes = extract_branch_codes(question)

    if not codes:
        return refusal("person_not_found", lang)

    items = []
    for code in codes:
        info = BRANCH_INFO.get(code, {})
        if lang == "th":
            province = info.get("province_th", "")
            name = info.get("th", code)
            if province and province != name:
                items.append(f"{code}: {name} จังหวัด{province}")
            else:
                items.append(f"{code}: {name}")
        else:
            name = info.get("en", code)
            items.append(f"{code}: {name}")

    return ", ".join(items)



def count_tool(row) -> str:
    lang = row["language"]
    question = row["question"]
    text = normalize_text(question)
    text_l = lower_text(question)
    text_u = upper_text(question)

    # 1) Explicit hyphen code from questions_wrangled.csv.
    hyphen_codes = row.get("hyphen_codes_list", [])
    if hyphen_codes:
        code = sorted([upper_text(c) for c in hyphen_codes], key=len, reverse=True)[0]
        hits = lookup_org_code_for_count(code)
        return format_count(len(hits), lang)

    # 2) Direct scan original section/dept codes from question text.
    for code in sorted(valid_section_original_codes, key=len, reverse=True):
        if re.search(rf"(?<![A-Z0-9]){re.escape(code)}(?![A-Z0-9])", text_u):
            hits = lookup_org_code_for_count(code)
            return format_count(len(hits), lang)

    for code in sorted(valid_department_original_codes, key=len, reverse=True):
        if re.search(rf"(?<![A-Z0-9]){re.escape(code)}(?![A-Z0-9])", text_u):
            hits = lookup_org_code_for_count(code)
            return format_count(len(hits), lang)

    # 3) Fallback org codes.
    org_codes = extract_org_codes(question, row)
    if org_codes:
        codes_with_hyphen = [c for c in org_codes if "-" in c]
        if codes_with_hyphen:
            code = sorted(codes_with_hyphen, key=len, reverse=True)[0]
        else:
            code = sorted(org_codes, key=len, reverse=True)[0]

        hits = lookup_org_code_for_count(code)
        return format_count(len(hits), lang)

    # 4) Branch.
    branch_codes = extract_branch_codes(question)
    if branch_codes:
        return format_count(len(by_branch.get(branch_codes[0], [])), lang)

    # 5) Surname.
    if "นามสกุล" in text:
        target = normalize_text(text.split("นามสกุล", 1)[1])
        target = re.sub(r"มีกี่คน.*", "", target).strip()
        if target:
            n = sum(1 for r in emp_records if target in r.get("Last Name Thai", ""))
            return format_count(n, lang)

    m = re.search(r"share the surname\s+(.+)$", text_l)
    if m:
        target = normalize_text(m.group(1))
        n = sum(
            1 for r in emp_records
            if target.lower() in lower_text(r.get("Last Name English", ""))
            or target in r.get("Last Name Thai", "")
        )
        return format_count(n, lang)

    # 6) First name.
    if "ชื่อ" in text or "คนชื่อ" in text:
        for first, records in sorted(by_first_name_th.items(), key=lambda kv: len(kv[0]), reverse=True):
            if first and first in text:
                return format_count(len(records), lang)

    # 7) Nickname.
    people = find_nickname_records(question)
    if people:
        return format_count(len(people), lang)

    people = find_people_in_question(question)
    if people:
        return format_count(len(people), lang)

    return format_count(0, lang)


def tier_listing_tool(row) -> str:
    lang = row["language"]
    ql = lower_text(row["question"])

    if "director" in ql or "รายชื่อ director" in ql:
        records = [r for r in emp_records if upper_text(r.get("Position Level", "")) == "DIRECTOR"]
        return format_people(records, lang=lang, max_items=None, contact=False)

    if "vp" in ql:
        records = [r for r in emp_records if upper_text(r.get("Position Level", "")) == "VP"]
        return format_people(records, lang=lang, max_items=None, contact=False)

    return refusal("person_not_found", lang)

def brand_lookup_tool(row) -> str:
    lang = row["language"]
    question = row["question"]
    ql = lower_text(question)
    bkey = extract_brand_key(question)

    # Retail network informal listing
    if "retail network" in ql:
        records = by_department.get("RET", [])
        return format_people(records, lang=lang, max_items=20, contact=False)

    if not bkey:
        return refusal("person_not_found", lang)

    data = BRAND_CODES[bkey]
    dept = data["dept"]
    gm_code = data["gm"]
    vp_code = data["vp"]

    if "ทีมเดียว" in ql:
        gm = by_unit.get(gm_code, [None])[0]
        if gm:
            section = upper_text(gm.get("Section_filled", ""))
            records = by_section.get(section, [])
            return format_people(records, lang=lang, max_items=None, contact=False)

    if "มีใครบ้าง" in ql or "who's on" in ql or "รายชื่อ" in ql:
        records = by_department.get(dept, [])
        return format_people(records, lang=lang, max_items=20, contact=False)

    if "gm" in ql or "general manager" in ql or "manages" in ql or "ดูแล" in ql:
        records = by_unit.get(gm_code, [])
        if records:
            return person_name(records[0], lang=lang, bilingual=True)

    if "vp" in ql or "ใครเป็น" in ql:
        records = by_unit.get(vp_code, [])
        if records:
            return person_name(records[0], lang=lang, bilingual=True)

    return refusal("person_not_found", lang)

def hierarchy_tool(row) -> str:
    lang = row["language"]
    ql = lower_text(row["question"])

    if "ใต้ cfo" in ql:
        records = []
        for code in ["FINVP", "CFO-EA"]:
            records.extend(by_unit.get(code, []))
        return format_people(records, lang=lang, max_items=None, contact=False)

    if "cpo ดูแล" in ql or "under cpo" in ql:
        records = []
        for code in ["CPO-EA", "SFVP", "DNVP", "KSVP", "WKVP", "JCVP"]:
            records.extend(by_unit.get(code, []))
        return format_people(records, lang=lang, max_items=None, contact=False)

    if "ceo-cos" in ql or "chief of staff" in ql:
        records = by_unit.get("CEO", [])
        if records:
            return person_name(records[0], lang=lang, bilingual=True)

    return refusal("person_not_found", lang)

def nickname_category_tool(row, category: str | None = None) -> str:
    lang = row["language"]
    ql = lower_text(row["question"])

    if category is None:
        if "ผลไม้" in ql:
            category = "fruit"
        elif "สี" in ql:
            category = "color"

    if category == "fruit":
        nicks_th = ["ส้ม", "พีช", "เปิ้ล"]
        nicks_en = ["som", "peach", "ple"]
    elif category == "color":
        nicks_th = ["ฟ้า", "ชมพู"]
        nicks_en = ["fah", "chompoo"]
    else:
        return refusal("person_not_found", lang)

    records = []
    for n in nicks_th:
        records.extend(by_nickname_th.get(n, []))
    for n in nicks_en:
        records.extend(by_nickname_en.get(n, []))

    records = list({r["Employee ID"]: r for r in records}.values())
    return format_people(records, lang=lang, max_items=None, contact=False)

def surname_family_tool(row) -> str:
    lang = row["language"]

    # Find repeated Thai last names.
    groups = defaultdict(list)
    for r in emp_records:
        ln = normalize_text(r.get("Last Name Thai", ""))
        if ln:
            groups[ln].append(r)

    repeated = [(ln, rs) for ln, rs in groups.items() if len(rs) >= 2]
    repeated = sorted(repeated, key=lambda x: (-len(x[1]), x[0]))

    if not repeated:
        return refusal("person_not_found", lang)

    # Return a few examples with same surname.
    parts = []
    for ln, rs in repeated[:3]:
        names = ", ".join(person_name(r, lang=lang, bilingual=True) for r in rs[:3])
        if lang == "th":
            parts.append(f"นามสกุล {ln}: {names}")
        else:
            parts.append(f"Surname {ln}: {names}")

    return "; ".join(parts)

def identity_lookup_tool(row) -> str:
    lang = row["language"]
    question = row["question"]
    ql = lower_text(question)

    if any(x in ql for x in ["ใต้ cfo", "cpo ดูแล", "ceo-cos"]):
        return hierarchy_tool(row)

    role_code = extract_role_code(question, row)
    if role_code and role_code in by_unit:
        return person_name(by_unit[role_code][0], lang=lang, bilingual=True)

    people = find_people_in_question(question)
    if not people:
        people = find_nickname_records(question)

    if not people:
        return refusal("person_not_found", lang)

    if any(x in ql for x in ["เบอร์", "ต่อ", "phone", "ext", "number"]):
        return format_people(people, lang=lang, max_items=3, contact=True)

    # For disambiguation questions like "ชื่อจริง X", include contact tokens to satisfy stricter labels.
    if "ชื่อจริง" in ql:
        return format_people(people, lang=lang, max_items=3, contact=True)

    return format_people(people, lang=lang, max_items=5, contact=False)


## Section 12 — Optional Typhoon v2.5 formatter

In [13]:

# ============================================================
# Section 12: Optional Typhoon v2.5 formatter
# ============================================================

# Keep False for the first deterministic baseline.
# If you enable it, use Typhoon v2.5 only.
USE_TYPHOON_FORMATTER = False

TYPHOON_MODEL = "typhoon-v2.5-30b-a3b-instruct"
TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY", "")
TYPHOON_API_URL = os.getenv("TYPHOON_API_URL", "")

def typhoon_format_answer(question: str, language: str, raw_answer: str) -> str:
    """
    Optional formatter.
    This cell intentionally does not hardcode a provider URL.
    Set TYPHOON_API_URL and TYPHOON_API_KEY in Colab environment if needed.

    Expected endpoint style: OpenAI-compatible /chat/completions.
    """
    if not USE_TYPHOON_FORMATTER:
        return raw_answer

    if not TYPHOON_API_KEY or not TYPHOON_API_URL:
        raise ValueError("Set TYPHOON_API_KEY and TYPHOON_API_URL before enabling Typhoon formatter.")

    import requests

    system_msg = (
        "You are a formatter for FahMai directory answers. "
        "Do not add facts. Do not change names, numbers, emails, IDs, or refusal phrases. "
        "Use the required language only."
    )
    user_msg = f"Question language: {language}\nQuestion: {question}\nRaw answer: {raw_answer}\nReturn final answer only."

    payload = {
        "model": TYPHOON_MODEL,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        "temperature": 0,
    }

    headers = {
        "Authorization": f"Bearer {TYPHOON_API_KEY}",
        "Content-Type": "application/json",
    }

    resp = requests.post(TYPHOON_API_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"].strip()



## Section 12.0 — Typhoon API credential setup

ก่อนรัน Section 12.1 ให้ตั้งค่า API key ด้วยวิธีใดวิธีหนึ่ง:

### วิธีแนะนำ: Colab Secrets

1. เปิดแถบ Secrets ใน Colab
2. เพิ่ม key ชื่อ:

```text
TYPHOON_API_KEY
```

3. ใส่ API key จาก OpenTyphoon

### หรือใช้ environment variable

```python
import os
os.environ["TYPHOON_API_KEY"] = "your_api_key_here"
```

Notebook นี้ตั้ง API URL ให้แล้ว:

```text
https://api.opentyphoon.ai/v1/chat/completions
```


In [14]:

# ============================================================
# Section 12.0: Typhoon API credential setup
# ============================================================

import os

# API URL for direct requests.post() chat completion call.
os.environ["TYPHOON_API_URL"] = "https://api.opentyphoon.ai/v1/chat/completions"

# Prefer Colab Secrets if available.
try:
    from google.colab import userdata
    secret_key = userdata.get("TYPHOON_API_KEY")
    if secret_key:
        os.environ["TYPHOON_API_KEY"] = secret_key
        print("Loaded TYPHOON_API_KEY from Colab Secrets.")
    else:
        print("TYPHOON_API_KEY not found in Colab Secrets.")
except Exception as e:
    print("Colab Secrets unavailable or not running in Colab:", repr(e))

# Check status without printing the key.
has_key = bool(os.getenv("TYPHOON_API_KEY", "").strip())
print("TYPHOON_API_KEY loaded =", has_key)
print("TYPHOON_API_URL =", os.getenv("TYPHOON_API_URL"))

if not has_key:
    print(
        "\n⚠️ Please add TYPHOON_API_KEY in Colab Secrets or set os.environ['TYPHOON_API_KEY'] before running Section 12.1."
    )


Loaded TYPHOON_API_KEY from Colab Secrets.
TYPHOON_API_KEY loaded = True
TYPHOON_API_URL = https://api.opentyphoon.ai/v1/chat/completions



## Section 12.1 — Typhoon v2.5 Agentic Harness

เปิดใช้เฉพาะเมื่อมี API key แล้ว:

```python
USE_TYPHOON_HARNESS = True
```

ค่า default ปิดไว้ก่อน:

```python
USE_TYPHOON_HARNESS = False
```

Harness นี้ให้ Typhoon ทำเฉพาะ refinement ในเคสที่ deterministic answer ยัง fail public label หรือเป็น route ที่เสี่ยง โดยจำกัด retry ไม่เกิน 2 รอบ


In [15]:

# ============================================================
# Section 12.1: Typhoon v2.5 Agentic Harness
# ============================================================

USE_TYPHOON_HARNESS = True
TYPHOON_MAX_RETRIES = 2

TYPHOON_MODEL = "typhoon-v2.5-30b-a3b-instruct"

# Load key from env; Section 12.0 can populate env from Colab Secrets.
TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY", "").strip()
TYPHOON_API_URL = os.getenv(
    "TYPHOON_API_URL",
    "https://api.opentyphoon.ai/v1/chat/completions"
).strip()

print("USE_TYPHOON_HARNESS =", USE_TYPHOON_HARNESS)
print("TYPHOON_MODEL =", TYPHOON_MODEL)
print("TYPHOON_API_URL =", TYPHOON_API_URL)
print("TYPHOON_API_KEY loaded =", bool(TYPHOON_API_KEY))

if USE_TYPHOON_HARNESS and not TYPHOON_API_KEY:
    raise ValueError(
        "USE_TYPHOON_HARNESS=True but TYPHOON_API_KEY is missing. "
        "Add TYPHOON_API_KEY in Colab Secrets or set os.environ['TYPHOON_API_KEY']."
    )

ALLOWED_TOOLS = [
    "refusal_tool",
    "extension_reverse_tool",
    "mobile_reverse_tool",
    "email_reverse_tool",
    "secretary_tool",
    "role_lookup_tool",
    "org_listing_tool",
    "contact_lookup_tool",
    "email_lookup_tool",
    "nickname_lookup_tool",
    "nickname_org_lookup_tool",
    "branch_lookup_tool",
    "count_tool",
    "tier_listing_tool",
    "brand_lookup_tool",
    "hierarchy_tool",
    "nickname_category_tool",
    "surname_family_tool",
    "identity_lookup_tool",
]

def call_typhoon_chat(messages: list[dict], temperature: float = 0.0, max_tokens: int = 512) -> str:
    """
    OpenAI-compatible request to OpenTyphoon.
    Competition rule: use Typhoon v2.5 only.
    """
    if not TYPHOON_API_KEY:
        raise ValueError("Missing TYPHOON_API_KEY")
    if not TYPHOON_API_URL:
        raise ValueError("Missing TYPHOON_API_URL")

    import requests

    payload = {
        "model": TYPHOON_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    headers = {
        "Authorization": f"Bearer {TYPHOON_API_KEY}",
        "Content-Type": "application/json",
    }

    response = requests.post(TYPHOON_API_URL, headers=headers, json=payload, timeout=90)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"].strip()

def compact_employee_row(rec: dict) -> dict:
    """
    Compact context row for Typhoon.
    Do not send full CSV into context.
    """
    return {
        "employee_id": rec.get("Employee ID", ""),
        "name_th": rec.get("full_name_th", ""),
        "name_en": rec.get("full_name_en", ""),
        "nickname_th": rec.get("Nickname Thai", ""),
        "nickname_en": rec.get("Nickname English", ""),
        "department": rec.get("Department_filled", ""),
        "section": rec.get("Section_filled", ""),
        "unit": rec.get("Unit_upper", ""),
        "position_th": rec.get("Position in Thai", ""),
        "position_en": rec.get("Position in English", ""),
        "email": rec.get("Email Address", ""),
        "phone_extension": rec.get("Phone Extension", ""),
        "mobile": rec.get("Mobile No.", ""),
        "branch": rec.get("Branch", ""),
        "office_location": rec.get("Office Location", ""),
        "start_year": rec.get("Start Year", ""),
        "position_level": rec.get("Position Level", ""),
    }

def retrieve_candidate_context(row, max_rows: int = 12) -> list[dict]:
    """
    Retrieve small candidate context for Typhoon.
    """
    question = row["question"]
    route = row["route_guess"]
    candidates = []

    if route == "refusal":
        return []

    # Reverse lookup context
    if "EMAIL_RE" in globals() and EMAIL_RE.search(question):
        m = EMAIL_RE.search(question)
        rec = by_email.get(lower_text(m.group(0))) if m else None
        return [compact_employee_row(rec)] if rec else []

    if "MOBILE_RE" in globals() and MOBILE_RE.search(question):
        m = MOBILE_RE.search(question)
        rec = by_mobile.get(m.group(0)) if m else None
        return [compact_employee_row(rec)] if rec else []

    if "EXT_RE" in globals() and EXT_RE.search(question):
        m = EXT_RE.search(question)
        rec = by_phone_ext.get(m.group(0)) if m else None
        if rec:
            return [compact_employee_row(rec)]

    # Role contexts
    try:
        role_codes = extract_all_role_codes(question, row)
    except Exception:
        role_codes = []
    for code in role_codes:
        candidates.extend(by_unit.get(code, []))

    # Org contexts
    try:
        org_codes = extract_org_codes(question, row)
    except Exception:
        org_codes = []
    for code in org_codes:
        candidates.extend(lookup_org_code(code)[:max_rows])

    # Person contexts
    try:
        candidates.extend(find_people_in_question(question))
    except Exception:
        pass
    try:
        candidates.extend(find_nickname_records(question))
    except Exception:
        pass

    # Branch contexts
    try:
        branch_codes = extract_branch_codes(question)
        for b in branch_codes:
            candidates.extend(by_branch.get(b, [])[:max_rows])
    except Exception:
        pass

    deduped = {}
    for rec in candidates:
        if rec and rec.get("Employee ID"):
            deduped[rec["Employee ID"]] = rec

    selected = list(deduped.values())[:max_rows]
    return [compact_employee_row(r) for r in selected]

def strict_language_check(answer: str, lang: str) -> bool:
    answer = normalize_text(answer)
    if not answer:
        return False

    thai_chars = len(re.findall(r"[\u0E00-\u0E7F]", answer))
    latin_chars = len(re.findall(r"[A-Za-z]", answer))

    if lang == "th":
        # Thai answer can include English names/emails.
        return thai_chars > 0 or latin_chars > 0
    return latin_chars > 0

def is_exact_refusal(answer: str) -> bool:
    phrases = []
    for v in REFUSAL_PHRASES.values():
        phrases.extend(v.values())
    return normalize_text(answer) in phrases

def sanitize_typhoon_answer(answer: str) -> str:
    ans = normalize_text(answer)
    ans = re.sub(r"^final answer\s*:\s*", "", ans, flags=re.IGNORECASE)
    ans = re.sub(r"^answer\s*:\s*", "", ans, flags=re.IGNORECASE)
    ans = ans.strip().strip('"').strip("'").strip()
    ans = " ".join(ans.splitlines()).strip()
    return ans


def grade_item(gt: dict, resp: str) -> tuple[bool, list[str]]:
    """
    Local token-based grader helper used by Typhoon Harness before Section 17.
    This mirrors the public error-analysis logic so the harness can decide
    whether a deterministic answer already passes before calling Typhoon.
    """
    ea = gt.get("expected_answer") or {}
    fails = []
    resp_l = str(resp).lower()

    for group in ea.get("must_contain_any_of", []):
        if group and not any(t and str(t).lower() in resp_l for t in group):
            fails.append(f"missing any-of {group[:3]}")

    for bad in ea.get("must_not_contain", []):
        if bad and str(bad).lower() in resp_l:
            fails.append(f"contains forbidden: {bad}")

    if ea.get("must_not_contain_phone_extension"):
        if re.search(r"\b\d{5}\b", str(resp)):
            fails.append("leaked phone extension")

    if ea.get("must_not_contain_employee_id_pattern"):
        if re.search(r"\b0[08]\d{6}\b", str(resp)):
            fails.append("leaked Employee ID")

    tokens_per_id = ea.get("all_items_tokens_per_id") or {}
    if tokens_per_id:
        matched_ids = []
        for emp_id, toks in tokens_per_id.items():
            if toks and any(t and str(t).lower() in resp_l for t in toks):
                matched_ids.append(emp_id)

        min_items = ea.get("min_items")
        if min_items is not None and len(matched_ids) < min_items:
            fails.append(f"min_items {len(matched_ids)}/{min_items}")

        exact_count = ea.get("exact_count")
        if exact_count is not None and len(matched_ids) != exact_count:
            fails.append(f"exact_count got {len(matched_ids)}, need {exact_count}")

    return len(fails) == 0, fails

def validate_against_public_label_if_available(question_id: str, answer: str) -> tuple[bool | None, list[str]]:
    """
    Uses public train labels only for local development feedback.
    """
    if "label_df" not in globals():
        return None, []

    matches = label_df[label_df["id"] == question_id]
    if len(matches) == 0:
        return None, []

    gt = next(item for item in train_labels["items"] if item["id"] == question_id)
    ok, fails = grade_item(gt, answer)
    return ok, fails

def typhoon_agentic_answer(row, deterministic_answer: str, deterministic_tool: str, feedback: str = "") -> tuple[str, str]:
    """
    Ask Typhoon to refine answer using only retrieved context and deterministic answer.
    """
    lang = row["language"]
    question = row["question"]

    # Never let LLM rewrite exact refusal phrases.
    if refusal_reason(row):
        return refusal(refusal_reason(row), lang), "rule_exact_refusal"

    context = retrieve_candidate_context(row)

    system_prompt = """
You are a FahMai directory QA agent.
Use ONLY the provided deterministic answer and retrieved employee context.
Do not use outside knowledge.
Do not invent names, phone numbers, emails, branches, or departments.
Do not output reasoning.
Return final free-text answer only.
If language is th, answer in Thai.
If language is en, answer in English.
If context is insufficient, return:
- th: ไม่พบข้อมูล
- en: no record found
"""

    user_payload = {
        "question_id": row["id"],
        "language": lang,
        "question": question,
        "route_guess": row["route_guess"],
        "deterministic_tool": deterministic_tool,
        "deterministic_answer": deterministic_answer,
        "retrieved_context": context,
        "allowed_tools": ALLOWED_TOOLS,
        "feedback_from_previous_attempt": feedback,
        "instruction": (
            "Return only the final answer. Keep names/numbers/emails exact. "
            "Do not output JSON. Do not add explanation."
        ),
    }

    messages = [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False, indent=2)},
    ]

    ans = call_typhoon_chat(messages, temperature=0.0, max_tokens=512)
    ans = sanitize_typhoon_answer(ans)

    if not strict_language_check(ans, lang):
        return deterministic_answer, "typhoon_rejected_language"

    return ans, "typhoon_agentic_harness"

def maybe_improve_with_typhoon(row, deterministic_answer: str, deterministic_tool: str) -> tuple[str, str, int]:
    """
    Retry no more than TYPHOON_MAX_RETRIES.
    """
    if not USE_TYPHOON_HARNESS:
        return deterministic_answer, deterministic_tool, 0

    if refusal_reason(row):
        return deterministic_answer, deterministic_tool, 0

    current = deterministic_answer
    current_tool = deterministic_tool

    label_ok, label_fails = validate_against_public_label_if_available(row["id"], current)

    # If deterministic already passes public label, do not call Typhoon.
    if label_ok is True:
        return current, current_tool, 0

    risky_routes = {
        "unknown",
        "identity_lookup",
        "nickname_lookup",
        "nickname_org_lookup",
        "brand_lookup",
        "hierarchy",
        "surname_family",
        "nickname_category_lookup",
    }

    # If no public label, call Typhoon only for risky routes or errors.
    if label_ok is None and row["route_guess"] not in risky_routes and not str(current_tool).startswith("error"):
        return current, current_tool, 0

    feedback = ""
    if label_fails:
        feedback = "Previous answer failed local token checks: " + "; ".join(label_fails)

    for i in range(TYPHOON_MAX_RETRIES):
        candidate, tool_name = typhoon_agentic_answer(row, current, current_tool, feedback=feedback)

        ok, fails = validate_against_public_label_if_available(row["id"], candidate)

        if ok is True:
            return candidate, tool_name, i + 1

        if ok is None and strict_language_check(candidate, row["language"]):
            return candidate, tool_name, i + 1

        feedback = "Candidate answer failed validation: " + "; ".join(fails or ["basic validation failed"])
        current = candidate
        current_tool = tool_name

    return deterministic_answer, deterministic_tool, TYPHOON_MAX_RETRIES


USE_TYPHOON_HARNESS = True
TYPHOON_MODEL = typhoon-v2.5-30b-a3b-instruct
TYPHOON_API_URL = https://api.opentyphoon.ai/v1/chat/completions
TYPHOON_API_KEY loaded = True


## Section 13 — Answer dispatcher

In [16]:

# ============================================================
# Section 13: Answer dispatcher
# ============================================================

def generate_answer_deterministic(row) -> tuple[str, str]:
    """
    Deterministic/rule-based answer only.
    Returns: (answer, tool_name)
    """
    route = row["route_guess"]

    if route == "refusal":
        return refusal_tool(row), "refusal_tool"

    if route == "extension_reverse_lookup":
        return extension_reverse_tool(row), "extension_reverse_tool"

    if route == "mobile_reverse_lookup":
        return mobile_reverse_tool(row), "mobile_reverse_tool"

    if route == "email_reverse_lookup":
        return email_reverse_tool(row), "email_reverse_tool"

    if route == "secretary_lookup":
        return secretary_tool(row), "secretary_tool"

    if route == "role_code_lookup":
        return role_lookup_tool(row), "role_lookup_tool"

    if route == "org_listing":
        return org_listing_tool(row), "org_listing_tool"

    if route == "contact_lookup":
        return contact_lookup_tool(row), "contact_lookup_tool"

    if route == "email_lookup":
        return email_lookup_tool(row), "email_lookup_tool"

    if route == "nickname_lookup":
        return nickname_lookup_tool(row), "nickname_lookup_tool"

    if route == "nickname_org_lookup":
        return nickname_org_lookup_tool(row), "nickname_org_lookup_tool"

    if route == "branch_lookup":
        return branch_lookup_tool(row), "branch_lookup_tool"

    if route == "count":
        return count_tool(row), "count_tool"

    if route == "tier_listing":
        return tier_listing_tool(row), "tier_listing_tool"

    if route == "brand_lookup":
        return brand_lookup_tool(row), "brand_lookup_tool"

    if route == "hierarchy":
        return hierarchy_tool(row), "hierarchy_tool"

    if route == "nickname_category_lookup":
        return nickname_category_tool(row), "nickname_category_tool"

    if route == "surname_family":
        return surname_family_tool(row), "surname_family_tool"

    if route == "identity_lookup":
        return identity_lookup_tool(row), "identity_lookup_tool"

    return identity_lookup_tool(row), "fallback_identity_lookup_tool"

def generate_answer(row) -> tuple[str, str]:
    """
    Harness-aware answer generation.
    Returns: (answer, tool_name)
    """
    lang = row["language"]

    try:
        deterministic_answer, deterministic_tool = generate_answer_deterministic(row)

        final_answer, final_tool, retry_count = maybe_improve_with_typhoon(
            row=row,
            deterministic_answer=deterministic_answer,
            deterministic_tool=deterministic_tool,
        )

        if retry_count > 0:
            final_tool = f"{final_tool}|retries={retry_count}"

        return final_answer.strip(), final_tool

    except Exception as e:
        return refusal("person_not_found", lang), f"error:{type(e).__name__}:{e}"

# Smoke test
for _, row in q.head(10).iterrows():
    ans, tool = generate_answer(row)
    print(row["id"], row["question"], "=>", ans, f"[{tool}]")


g001 who is the RETVP => Wiriya Chanchai (วิริยะ จันทชัย) [role_lookup_tool]
g002 ใครเป็น OPSVP => คึกฤทธิ์ บุษราคัมวงศ์ (Kukrit Busarakhamwong) [role_lookup_tool]
g004 LEGVP ใคร => ไพโรจน์ มหากุล (Phairoj Mahakun) [role_lookup_tool]
g005 ใครเป็น COO ตอนนี้ => พงษ์กานต์ ราชชากัญญ์ (Pongkan Rajchakan) [role_lookup_tool]
g007 ใครเป็น LOGFL ตอนนี้ => มาลี อมรทอง (Malee Amonthong) [role_lookup_tool]
g008 ใครเป็น DNVP ตอนนี้ => เรืองศักดิ์ เทพเกียรติกำจร (Ruangsak Thepkiatkamjorn) [role_lookup_tool]
g009 SUPVP ใคร => ดาริกา อาวุทธ์ดี (Darika Awutdi) [role_lookup_tool]
g011 who is the CHRO => Nathamon Aphichaidee (ณฐามน อภิชัยดี) [role_lookup_tool]
g012 ใครเป็น MKTVP => คะวัง กอบสุขรัตน์ (Kwang Kobsookrat) [role_lookup_tool]
g014 LOGVP ตอนนี้ใคร => ณัฐกานต์ ศรีอารมณ์ดี (Natthakan Sriaromdee) [role_lookup_tool]


## Section 14 — Generate submission

In [17]:

# ============================================================
# Section 14: Generate submission
# ============================================================

answers = {}
debug_rows = []

for _, row in q.iterrows():
    ans, tool = generate_answer(row)
    answers[row["id"]] = ans

    debug_rows.append({
        "id": row["id"],
        "language": row["language"],
        "question": row["question"],
        "route_guess": row["route_guess"],
        "route_guess_original": row.get("route_guess_original", ""),
        "refusal_reason_guess": row.get("refusal_reason_guess", ""),
        "refusal_reason_runtime": row.get("refusal_reason_runtime", ""),
        "tool": tool,
        "response": ans,
        "use_typhoon_harness": USE_TYPHOON_HARNESS,
    })

submission = sample[["id"]].copy()
submission["response"] = submission["id"].map(answers).fillna("")

debug_df = pd.DataFrame(debug_rows)

# Final checks
assert len(submission) == 300
assert list(submission.columns) == ["id", "response"]
assert submission["id"].is_unique
assert (submission["response"].str.strip() == "").sum() == 0, "Some responses are blank"

display(submission.head(20))
display(debug_df.head(20))

print("Blank responses:", (submission["response"].str.strip() == "").sum())
print("Tool distribution:")
display(debug_df["tool"].value_counts())


,id,response
0,g001,Wiriya Chanchai (วิริยะ จันทชัย)
1,g002,คึกฤทธิ์ บุษราคัมวงศ์ (Kukrit Busarakhamwong)
2,g004,ไพโรจน์ มหากุล (Phairoj Mahakun)
3,g005,พงษ์กานต์ ราชชากัญญ์ (Pongkan Rajchakan)
4,g007,มาลี อมรทอง (Malee Amonthong)
5,g008,เรืองศักดิ์ เทพเกียรติกำจร (Ruangsak Thepkiatkamjorn)
6,g009,ดาริกา อาวุทธ์ดี (Darika Awutdi)
7,g011,Nathamon Aphichaidee (ณฐามน อภิชัยดี)
8,g012,คะวัง กอบสุขรัตน์ (Kwang Kobsookrat)
9,g014,ณัฐกานต์ ศรีอารมณ์ดี (Natthakan Sriaromdee)


,id,language,question,route_guess,route_guess_original,refusal_reason_guess,refusal_reason_runtime,tool,response,use_typhoon_harness
0,g001,en,who is the RETVP,role_code_lookup,role_code_lookup,,,role_lookup_tool,Wiriya Chanchai (วิริยะ จันทชัย),True
1,g002,th,ใครเป็น OPSVP,role_code_lookup,role_code_lookup,,,role_lookup_tool,คึกฤทธิ์ บุษราคัมวงศ์ (Kukrit Busarakhamwong),True
2,g004,th,LEGVP ใคร,role_code_lookup,role_code_lookup,,,role_lookup_tool,ไพโรจน์ มหากุล (Phairoj Mahakun),True
3,g005,th,ใครเป็น COO ตอนนี้,role_code_lookup,role_code_lookup,,,role_lookup_tool,พงษ์กานต์ ราชชากัญญ์ (Pongkan Rajchakan),True
4,g007,th,ใครเป็น LOGFL ตอนนี้,role_code_lookup,role_code_lookup,,,role_lookup_tool,มาลี อมรทอง (Malee Amonthong),True
5,g008,th,ใครเป็น DNVP ตอนนี้,role_code_lookup,role_code_lookup,,,role_lookup_tool,เรืองศักดิ์ เทพเกียรติกำจร (Ruangsak Thepkiatkamjorn),True
6,g009,th,SUPVP ใคร,role_code_lookup,role_code_lookup,,,role_lookup_tool,ดาริกา อาวุทธ์ดี (Darika Awutdi),True
7,g011,en,who is the CHRO,role_code_lookup,role_code_lookup,,,role_lookup_tool,Nathamon Aphichaidee (ณฐามน อภิชัยดี),True
8,g012,th,ใครเป็น MKTVP,role_code_lookup,role_code_lookup,,,role_lookup_tool,คะวัง กอบสุขรัตน์ (Kwang Kobsookrat),True
9,g014,th,LOGVP ตอนนี้ใคร,role_code_lookup,role_code_lookup,,,role_lookup_tool,ณัฐกานต์ ศรีอารมณ์ดี (Natthakan Sriaromdee),True


Blank responses: 0
Tool distribution:


,count
tool,
role_lookup_tool,74
contact_lookup_tool,49
org_listing_tool,32
refusal_tool,27
typhoon_agentic_harness|retries=1,20
secretary_tool,20
count_tool,20
nickname_lookup_tool,10
brand_lookup_tool,9


## Section 15 — Save submission

In [18]:

# ============================================================
# Section 15: Save submission
# ============================================================

SUBMIT_DIR.mkdir(parents=True, exist_ok=True)

# Save main Kaggle submission file
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8-sig")

# Save debug file
debug_df.to_csv(DEBUG_PATH, index=False, encoding="utf-8-sig")

print("Saved Kaggle submission:", SUBMISSION_PATH)
print("exists =", SUBMISSION_PATH.exists(), "size =", SUBMISSION_PATH.stat().st_size if SUBMISSION_PATH.exists() else None)

print("Saved debug:", DEBUG_PATH)
print("exists =", DEBUG_PATH.exists(), "size =", DEBUG_PATH.stat().st_size if DEBUG_PATH.exists() else None)

print("\nSubmit folder contents:")
for p in sorted(SUBMIT_DIR.iterdir()):
    print("-", p.name, p.stat().st_size if p.is_file() else "")


Saved Kaggle submission: /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Submit/submission.csv
exists = True size = 87629
Saved debug: /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Submit/submission_debug_details_v6.csv
exists = True size = 119907

Submit folder contents:
- public_error_analysis_v6.csv 37416
- submission.csv 87629
- submission_debug_details_v6.csv 119907
- submission_no API.csv 88202


## Section 16 — Run local grader

In [19]:

# ============================================================
# Section 16: Run local grader
# ============================================================

cmd = ["python", str(GRADE_PATH), str(SUBMISSION_PATH), str(TRAIN_LABELS_PATH)]
print("Running:", " ".join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"grade.py failed with return code {result.returncode}")


Running: python /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/grade.py /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Submit/submission.csv /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/train_labels.json
STDOUT:
Scored 158 items against /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/train_labels.json
Passed: 158/158 = 100.0%

Bucket                             pass/total    rate
--------------------------------------------------------
nickname_grid                    17/      17  100.0%
refuse                           15/      15  100.0%
evp_secretary                    9/       9  100.0%
vp_identity                      9/       9  100.0%
casual_name_lookup               9/       9  100.0%
evp_identity_by_code             8/       8  100.0%
evp_identity_by_description      8/       8  100.0%
name_lookup                      8/       8  100.0%
dept_listing_medium              8/       8  100.0%
dept_member_count                7/      

## Section 17 — Public label error analysis

In [20]:

# ============================================================
# Section 17: Public label error analysis
# ============================================================

# Re-implement grader logic to inspect failed public items in notebook.

def grade_item(gt: dict, resp: str) -> tuple[bool, list[str]]:
    ea = gt.get("expected_answer") or {}
    fails = []
    resp_l = resp.lower()

    for group in ea.get("must_contain_any_of", []):
        if group and not any(t and t.lower() in resp_l for t in group):
            fails.append(f"missing any-of {group[:3]}")

    for bad in ea.get("must_not_contain", []):
        if bad and bad.lower() in resp_l:
            fails.append(f"contains forbidden: {bad}")

    if ea.get("must_not_contain_phone_extension"):
        if re.search(r"\b\d{5}\b", resp):
            fails.append("leaked phone extension")

    if ea.get("must_not_contain_employee_id_pattern"):
        if re.search(r"\b0[08]\d{6}\b", resp):
            fails.append("leaked Employee ID")

    tokens_per_id = ea.get("all_items_tokens_per_id") or {}
    if tokens_per_id:
        matched_ids = []
        for emp_id, toks in tokens_per_id.items():
            if toks and any(t and t.lower() in resp_l for t in toks):
                matched_ids.append(emp_id)

        min_items = ea.get("min_items")
        if min_items is not None and len(matched_ids) < min_items:
            fails.append(f"min_items {len(matched_ids)}/{min_items}")

        exact_count = ea.get("exact_count")
        if exact_count is not None and len(matched_ids) != exact_count:
            fails.append(f"exact_count got {len(matched_ids)}, need {exact_count}")

    return len(fails) == 0, fails

sub_map = dict(zip(submission["id"], submission["response"]))

analysis_rows = []
for gt in train_labels["items"]:
    iid = gt["id"]
    resp = sub_map.get(iid, "")
    ok, fails = grade_item(gt, resp)

    qrow = q[q["id"] == iid].iloc[0].to_dict()
    drow = debug_df[debug_df["id"] == iid].iloc[0].to_dict()

    analysis_rows.append({
        "id": iid,
        "passed": ok,
        "bucket": gt.get("bucket", ""),
        "priority": gt.get("priority", ""),
        "language": gt.get("language", ""),
        "question": gt.get("question", ""),
        "route_guess": drow.get("route_guess", ""),
        "tool": drow.get("tool", ""),
        "response": resp,
        "fails": "; ".join(fails),
    })

analysis_df = pd.DataFrame(analysis_rows)

print("Public pass rate:")
display(analysis_df["passed"].value_counts())

print("Bucket pass rate:")
bucket_rate = (
    analysis_df
    .groupby("bucket")
    .agg(total=("id", "count"), passed=("passed", "sum"))
    .assign(rate=lambda d: d["passed"] / d["total"])
    .sort_values("rate")
)
display(bucket_rate)

print("Failed items:")
display(analysis_df[~analysis_df["passed"]].head(100))

SUBMIT_DIR.mkdir(parents=True, exist_ok=True)
analysis_df.to_csv(ERROR_ANALYSIS_PATH, index=False, encoding="utf-8-sig")
print("Saved:", ERROR_ANALYSIS_PATH)


Public pass rate:


,count
passed,
True,158


Bucket pass rate:


,total,passed,rate
bucket,,,
casual_name_lookup,9,9,1.0
ceo_president,3,3,1.0
dept_listing_medium,8,8,1.0
dept_listing_small,6,6,1.0
dept_member_count,7,7,1.0
email_identity_lookup,2,2,1.0
email_mobile_lookup,4,4,1.0
evp_identity_by_code,8,8,1.0
evp_identity_by_description,8,8,1.0


Failed items:


,id,passed,bucket,priority,language,question,route_guess,tool,response,fails


Saved: /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Submit/public_error_analysis_v6.csv



## Section 18 — Next iteration guide

หลังรัน grader แล้ว ให้ปรับตาม bucket ที่ fail มากที่สุด

ลำดับแนะนำ:
1. `refuse` — ต้อง phrase เป๊ะ ห้าม leak phone/employee id
2. `evp_identity_by_code` / `vp_identity` — role code lookup
3. `evp_secretary` — เลขา / EA / secretary
4. `name_lookup` / `casual_name_lookup` — ชื่อ, ชื่อเล่น, department context
5. `nickname_grid` — nickname หลายคน, count, variant
6. `dept_listing_*` / `dept_member_count` — org listing/count
7. reporting chain / branch / hard private-like questions

ไฟล์ที่ใช้ debug:
```text
submission_debug_details.csv
public_error_analysis.csv
```



## Section 18.1 — Verify files in Submit folder

ตรวจว่าไฟล์ submission ถูกบันทึกในโฟลเดอร์ `Submit` เรียบร้อยแล้ว


In [21]:

# ============================================================
# Section 18.1: Verify files in Submit folder
# ============================================================

print("Submit folder:", SUBMIT_DIR)
print("exists =", SUBMIT_DIR.exists())

expected_files = [
    SUBMISSION_PATH,
    DEBUG_PATH,
    ERROR_ANALYSIS_PATH,
]

for p in expected_files:
    print(p.name, "exists =", p.exists(), "size =", p.stat().st_size if p.exists() else None)

# Reload submission and validate Kaggle format
submission_check = pd.read_csv(SUBMISSION_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")

print("\nsubmission.csv shape:", submission_check.shape)
print("columns:", list(submission_check.columns))
print("blank responses:", (submission_check["response"].str.strip() == "").sum())
print("id unique:", submission_check["id"].is_unique)

assert submission_check.shape == (300, 2)
assert list(submission_check.columns) == ["id", "response"]
assert submission_check["id"].is_unique
assert (submission_check["response"].str.strip() == "").sum() == 0

print("\n✅ Ready to upload to Kaggle:", SUBMISSION_PATH)


Submit folder: /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Submit
exists = True
submission.csv exists = True size = 87629
submission_debug_details_v6.csv exists = True size = 119907
public_error_analysis_v6.csv exists = True size = 69686

submission.csv shape: (300, 2)
columns: ['id', 'response']
blank responses: 0
id unique: True

✅ Ready to upload to Kaggle: /content/drive/MyDrive/super-ai-engineer-season-6-fahmai-2/Submit/submission.csv



## Section 18.2 — Confirm Typhoon usage

หลังรันจบ ให้เปิดไฟล์:

```text
Submit/submission_debug_details_v6.csv
```

เช็กคอลัมน์:

```text
use_typhoon_harness
tool
```

ถ้าเปิด Typhoon สำเร็จ:

- `use_typhoon_harness` ต้องเป็น `True`
- บางแถวใน `tool` อาจมีค่าเช่น `typhoon_agentic_harness|retries=1`

หมายเหตุ: Harness จะไม่เรียก Typhoon ในข้อที่ deterministic answer ผ่าน public validation แล้ว เพื่อไม่ทำให้คำตอบที่ผ่านแล้วเสีย
